#White Box Model

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
import io
from google.colab import files
from sklearn.impute import SimpleImputer # This import is already present in kOCKlqywmBRd, adding it here for completeness of the cell content.

print("Libraries imported successfully!")

# --- 2.1 Manual Dataset Upload ---
uploaded = files.upload()

# Assuming only one file is uploaded for the dataset
for fn in uploaded.keys():
  uploaded_file_path = fn
  print(f'User uploaded file "{uploaded_file_path}" (size: {len(uploaded[fn])} bytes)')

# --- 2. 🔥 Fairness Evaluation Engine (Core Module) ---
class FairnessEvaluationEngine:
    def __init__(self, protected_attribute_name='Gender', positive_label=1):
        self.protected_attribute_name = protected_attribute_name
        self.positive_label = positive_label

    def detect_potential_sensitive_attributes(self, df, exclude_cols=['Selected'], target_col=None):
        """Automatically suggests columns that might be sensitive attributes."""
        potential = []
        # Add the target_col to exclude_cols if it's provided
        if target_col and target_col not in exclude_cols:
            exclude_cols.append(target_col)

        for col in df.columns:
            if col in exclude_cols: continue
            # Heuristic: object/category types or numeric with very few unique values
            if df[col].dtype == 'object' or df[col].dtype.name == 'category' or df[col].nunique() < 10:
                potential.append(col)
        return potential

    def _get_metrics_by_group(self, y_true, y_pred, sensitive_attr):
        df = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'sensitive_attr': sensitive_attr})
        groups = sensitive_attr.unique()
        group_metrics = {}

        for group in groups:
            group_df = df[df['sensitive_attr'] == group]
            total_group_instances = len(group_df)
            if total_group_instances == 0: continue

            selection_rate = (group_df['y_pred'] == self.positive_label).sum() / total_group_instances
            tp = ((group_df['y_true'] == self.positive_label) & (group_df['y_pred'] == self.positive_label)).sum()
            fp = ((group_df['y_true'] != self.positive_label) & (group_df['y_pred'] == self.positive_label)).sum()
            tn = ((group_df['y_true'] != self.positive_label) & (group_df['y_pred'] != self.positive_label)).sum()
            fn = ((group_df['y_true'] == self.positive_label) & (group_df['y_pred'] != self.positive_label)).sum()

            # Handle division by zero for recall/TPR and FPR if denominator is zero
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
            fnr = fn / (fn + tp) if (fn + tp) > 0 else 0 # Calculate False Negative Rate

            accuracy = (tp + tn) / total_group_instances

            group_metrics[group] = {
                'selection_rate': selection_rate,
                'true_positive_rate': tpr,
                'false_positive_rate': fpr,
                'false_negative_rate': fnr, # Add FNR to metrics
                'accuracy': accuracy,
                'count': total_group_instances
            }
        return group_metrics

    def calculate_all_metrics(self, y_true, y_pred, sensitive_attr):
        group_metrics = self._get_metrics_by_group(y_true, y_pred, sensitive_attr)

        # Extract metrics for comparison
        selection_rates = {k: m['selection_rate'] for k, m in group_metrics.items()}
        tprs = {k: m['true_positive_rate'] for k, m in group_metrics.items()}
        fprs = {k: m['false_positive_rate'] for k, m in group_metrics.items()}
        fnrs = {k: m['false_negative_rate'] for k, m in group_metrics.items()}

        all_selection_rates = list(selection_rates.values())
        all_tprs = list(tprs.values())
        all_fprs = list(fprs.values())
        all_fnrs = list(fnrs.values())

        num_groups = len(all_selection_rates)

        # Initialize new metrics to 0 or default values
        demographic_parity_difference = 0.0
        disparate_impact = 1.0
        equal_opportunity_difference = 0.0
        equalized_odds_difference = 0.0
        selection_rate_difference = 0.0
        fpr_difference = 0.0
        fnr_difference = 0.0
        average_odds_difference = 0.0
        statistical_parity_difference = 0.0

        if num_groups > 1:
            # Identify privileged and unprivileged groups based on selection rate
            privileged_group_name = max(selection_rates, key=selection_rates.get)
            unprivileged_group_name = min(selection_rates, key=selection_rates.get)

            privileged_sr = selection_rates[privileged_group_name]
            unprivileged_sr = selection_rates[unprivileged_group_name]

            privileged_tpr = tprs[privileged_group_name]
            unprivileged_tpr = tprs[unprivileged_group_name]

            privileged_fpr = fprs[privileged_group_name]
            unprivileged_fpr = fprs[unprivileged_group_name]

            privileged_fnr = fnrs[privileged_group_name]
            unprivileged_fnr = fnrs[unprivileged_group_name]

            # 1. Demographic Parity Difference (DPD)
            demographic_parity_difference = max(all_selection_rates) - min(all_selection_rates)

            # 2. Disparate Impact (DI)
            disparate_impact = min(all_selection_rates) / max(all_selection_rates) if max(all_selection_rates) > 0 else 1

            # 3. Equal Opportunity Difference (EOD)
            equal_opportunity_difference = max(all_tprs) - min(all_tprs)

            # 4. Equalized Odds Difference (EODd)
            equalized_odds_tpr_diff = max(all_tprs) - min(all_tprs)
            equalized_odds_fpr_diff = max(all_fprs) - min(all_fprs)
            equalized_odds_difference = max(equalized_odds_tpr_diff, equalized_odds_fpr_diff)

            # --- New Metrics ---
            # 1. Selection Rate Difference (Privileged - Unprivileged)
            selection_rate_difference = privileged_sr - unprivileged_sr

            # 2. False Positive Rate Difference (Privileged - Unprivileged)
            fpr_difference = privileged_fpr - unprivileged_fpr

            # 3. False Negative Rate Difference (Privileged - Unprivileged)
            fnr_difference = privileged_fnr - unprivileged_fnr

            # 4. Average Odds Difference
            average_odds_difference = 0.5 * ((privileged_tpr - unprivileged_tpr) + (privileged_fpr - unprivileged_fpr))

            # 5. Statistical Parity Difference (Unprivileged - Privileged)
            statistical_parity_difference = unprivileged_sr - privileged_sr

        return {
            'group_metrics': group_metrics,
            'demographic_parity_difference': demographic_parity_difference,
            'disparate_impact': disparate_impact,
            'equal_opportunity_difference': equal_opportunity_difference,
            'equalized_odds_difference': equalized_odds_difference,
            # New metrics
            'selection_rate_difference': selection_rate_difference,
            'false_positive_rate_difference': fpr_difference,
            'false_negative_rate_difference': fnr_difference,
            'average_odds_difference': average_odds_difference,
            'statistical_parity_difference': statistical_parity_difference
        }

    def plot_bias_metrics(self, results, title="Bias Metrics", print_report=True):
        metrics_df = pd.DataFrame(results['group_metrics']).T
        # Plot selection rate, true positive rate, false positive rate, and false negative rate for each group
        fig, axes = plt.subplots(1, 4, figsize=(24, 6), sharey=True) # Increased subplots to 4 for FNR

        metrics_df['selection_rate'].plot(kind='bar', ax=axes[0], color='skyblue')
        axes[0].set_title('Selection Rate by Group')
        axes[0].set_ylabel('Rate')
        axes[0].tick_params(axis='x', rotation=45)

        metrics_df['true_positive_rate'].plot(kind='bar', ax=axes[1], color='lightcoral')
        axes[1].set_title('True Positive Rate (Equal Opportunity) by Group')
        axes[1].tick_params(axis='x', rotation=45)

        metrics_df['false_positive_rate'].plot(kind='bar', ax=axes[2], color='lightgreen')
        axes[2].set_title('False Positive Rate (Equalized Odds) by Group')
        axes[2].tick_params(axis='x', rotation=45)

        metrics_df['false_negative_rate'].plot(kind='bar', ax=axes[3], color='gold') # Plot FNR
        axes[3].set_title('False Negative Rate by Group')
        axes[3].tick_params(axis='x', rotation=45)

        plt.suptitle(title, fontsize=16, y=1.02)
        plt.tight_layout()
        plt.show()

        # Display summary differences
        if print_report:
            print(f"\nSummary for {title}:")
            print(f"  Demographic Parity Difference: {results['demographic_parity_difference']:.4f}")
            print(f"  Disparate Impact (min/max ratio): {results['disparate_impact']:.4f}")
            print(f"  Equal Opportunity Difference (TPR max-min): {results['equal_opportunity_difference']:.4f}")
            print(f"  Equalized Odds Difference (max of TPR/FPR diffs): {results['equalized_odds_difference']:.4f}")

            # --- New Fairness Metrics Report ---
            print("\n## Fairness Metrics Report")
            print(f"Selection Rate Difference: {results['selection_rate_difference']:.4f}")
            print(f"FPR Difference: {results['false_positive_rate_difference']:.4f}")
            print(f"FNR Difference: {results['false_negative_rate_difference']:.4f}")
            print(f"Average Odds Difference: {results['average_odds_difference']:.4f}")
            print(f"Statistical Parity Difference: {results['statistical_parity_difference']:.4f}")

# --- 3. Modular Code for White-Box Branch (Dataset-Based Analysis) ---
def run_white_box_branch(data_size=200, csv_file_path=None):
    print("\n--- Running White-Box Branch ---")

    # 1. Load/Generate Dataset
    if csv_file_path:
        print(f"Loading dataset from {csv_file_path}...")
        data = pd.read_csv(io.BytesIO(uploaded[csv_file_path]))
        # Clean column names to avoid hidden space errors
        data.columns = data.columns.str.strip()
        print("Dataset loaded successfully!")
    else:
        np.random.seed(42)
        num_features = 4
        data = pd.DataFrame({
            f'Feature_{i}': np.random.randint(0, 100, data_size) for i in range(num_features)
        })
        data['Sensitive_Attr_A'] = np.random.choice([0, 1], data_size)
        logit = 0.1 * data['Feature_0'] + 0.05 * data['Feature_1'] - 2.0 * data['Sensitive_Attr_A']
        prob = 1 / (1 + np.exp(-logit))
        data['Selected'] = (prob > 0.5).astype(int)

    # Robust target identification
    target_candidates = ['Selected', 'Selection', 'Label', 'Target', 'SelectionStatus']
    target_col = None

    for candidate in target_candidates:
        if candidate in data.columns:
            target_col = candidate
            break

    if not target_col:
        target_col = data.columns[-1]
        print(f"Warning: Standard target columns not found. Using '{target_col}' as target.")
    else:
        print(f"Using '{target_col}' as the target variable.")

    X = data.drop(target_col, axis=1)
    y = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize a dictionary to store LabelEncoders
    label_encoders = {}

    # Identify categorical columns (object or category dtype)
    initial_categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

    # Identify numeric columns with few unique values that should be treated as categorical
    numeric_as_categorical_cols = []
    for col in X_train.columns:
        if X_train[col].dtype in ['int64', 'float64'] and col not in initial_categorical_cols and X_train[col].nunique() < 10:
            numeric_as_categorical_cols.append(col)

    # Convert numeric-as-categorical columns to object dtype *before* imputation and encoding
    for col in numeric_as_categorical_cols:
        X_train[col] = X_train[col].astype(str)
        X_test[col] = X_test[col].astype(str)

    # Now, redefine categorical_cols to include all columns to be treated as categorical strings
    categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

    # Impute and LabelEncode categorical columns
    for col in categorical_cols:
        imputer_cat = SimpleImputer(strategy='constant', fill_value='missing')
        # Apply imputer on both train and test, fit only on train
        X_train[col] = imputer_cat.fit_transform(X_train[[col]]).ravel()
        X_test[col] = imputer_cat.transform(X_test[[col]]).ravel()

        le = LabelEncoder()
        # Ensure all values are strings before encoding, to handle potential mixed types if any
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))
        X_train[col] = X_train[col].astype('category')
        X_test[col] = X_test[col].astype('category')
        label_encoders[col] = le # Store the fitted LabelEncoder

    # Identify numerical columns (excluding those treated as categorical)
    # This should now correctly exclude columns converted to 'object' dtype
    numerical_cols = [c for c in X_train.select_dtypes(include=[np.number]).columns if c not in categorical_cols]

    # Impute numerical columns before scaling
    imputer_num = SimpleImputer(strategy='median')

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()

    if numerical_cols:
        # Impute numerical columns
        X_train_scaled[numerical_cols] = imputer_num.fit_transform(X_train[numerical_cols])
        X_test_scaled[numerical_cols] = imputer_num.transform(X_test[numerical_cols])

        # Then scale the imputed numerical columns
        scaler = StandardScaler()
        X_train_scaled[numerical_cols] = scaler.fit_transform(X_train_scaled[numerical_cols])
        X_test_scaled[numerical_cols] = scaler.transform(X_test_scaled[numerical_cols])

    print("Training XGBoost Model...")
    xgb_model = xgb.XGBClassifier(eval_metric='logloss', random_state=42, enable_categorical=True)
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)

    print("Training MLP Model...")
    X_train_mlp = X_train_scaled.select_dtypes(exclude=['category'])
    X_test_mlp = X_test_scaled.select_dtypes(exclude=['category'])

    # Final safeguard: Impute any remaining NaNs in the MLP dataframes just before training
    imputer_mlp_final = SimpleImputer(strategy='median') # Using median for numerical data
    if X_train_mlp.isnull().sum().sum() > 0:
        X_train_mlp = pd.DataFrame(imputer_mlp_final.fit_transform(X_train_mlp), columns=X_train_mlp.columns, index=X_train_mlp.index)
        X_test_mlp = pd.DataFrame(imputer_mlp_final.transform(X_test_mlp), columns=X_test_mlp.columns, index=X_test_mlp.index)

    mlp_model = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42, early_stopping=True)
    mlp_model.fit(X_train_mlp, y_train)
    y_pred_mlp = mlp_model.predict(X_test_mlp)

    print("White-Box Branch complete.")
    # Return the processed X_train_mlp and X_test_mlp for SHAP analysis
    return X_test, y_test, y_pred_xgb, y_pred_mlp, xgb_model, mlp_model, X_train, y_train, StandardScaler() if not numerical_cols else scaler, label_encoders, X_train_mlp, X_test_mlp

# --- Initial detection of sensitive attributes from uploaded file ---
engine = FairnessEvaluationEngine()

# Load the uploaded data to inspect columns
df_inspect = pd.read_csv(io.BytesIO(uploaded[uploaded_file_path]))
df_inspect.columns = df_inspect.columns.str.strip()

# Refresh the suggestions variable based on your new file
# First, run the white-box branch to identify the target column
X_test_wb, y_test_wb, y_pred_xgb_wb, y_pred_mlp_wb, xgb_model_wb, mlp_model_wb, X_train_wb, y_train_wb, scaler_wb, label_encoders_wb, X_train_mlp_processed, X_test_mlp_processed = run_white_box_branch(csv_file_path=uploaded_file_path)

# Then, use the identified target column to exclude it from sensitive attributes
target_col_name = y_test_wb.name # Get the name of the target column
suggestions = engine.detect_potential_sensitive_attributes(df_inspect, target_col=target_col_name)

print(f"File: {uploaded_file_path}")
print(f"Detected Attributes for Analysis: {suggestions}")

# --- 4. Automated Bias Suggestion and Attribute Selection ---

# 1. Automatic Bias Ranking Engine
def get_bias_suggestions(X_test, y_pred, suggestions):
    report = []
    engine = FairnessEvaluationEngine()

    # Define weights for each metric. These are example weights and should be tuned.
    # Weights should sum to 1 if you want the composite score to be easily comparable.
    metric_weights = {
        'demographic_parity_difference': 0.2,
        'disparate_impact': 0.2, # Will be transformed to abs(1 - DI)
        'equal_opportunity_difference': 0.2,
        'selection_rate_difference': 0.1,
        'false_positive_rate_difference': 0.1,
        'false_negative_rate_difference': 0.1,
        'average_odds_difference': 0.05,
        'statistical_parity_difference': 0.05
    }

    # Ensure weights sum to 1 or normalize them if they don't
    total_weight = sum(metric_weights.values())
    if total_weight != 1.0:
        metric_weights = {k: v / total_weight for k, v in metric_weights.items()}

    for attr in suggestions:
        try:
            # Ensure the attribute column is present in X_test before calculating metrics
            if attr not in X_test.columns:
                print(f"Skipping bias calculation for attribute {attr}: Not found in X_test.")
                continue

            res = engine.calculate_all_metrics(y_test_wb, y_pred, X_test[attr])
            composite_bias_score = 0.0

            for metric_name, weight in metric_weights.items():
                metric_value = res.get(metric_name, 0.0)

                # Apply normalization based on metric type
                normalized_metric_value = 0.0
                if 'difference' in metric_name: # For difference metrics (0 is ideal)
                    normalized_metric_value = abs(metric_value)
                elif metric_name == 'disparate_impact': # For ratio metrics (1 is ideal)
                    normalized_metric_value = abs(1.0 - metric_value)
                # Add other normalization rules if new metric types are introduced

                composite_bias_score += weight * normalized_metric_value

            report.append({'Attribute': attr, 'Bias_Score': composite_bias_score})
        except Exception as e:
            print(f"Could not calculate composite bias score for attribute {attr}: {e}")
            continue
    return pd.DataFrame(report).sort_values(by='Bias_Score', ascending=False) if report else pd.DataFrame(columns=['Attribute', 'Bias_Score'])

# Removed the first call to run_white_box_branch here
# It's now called earlier to get the target_col_name before suggestions are detected

# Re-initialize Engine and Detect Attributes to ensure 'suggestions' is defined
engine_for_suggestions = FairnessEvaluationEngine()
# Load the uploaded data to inspect columns
df_inspect = pd.read_csv(io.BytesIO(uploaded[uploaded_file_path]))
df_inspect.columns = df_inspect.columns.str.strip()
# Refresh the suggestions variable based on your new file, ensuring target_col_name is excluded
suggestions = engine_for_suggestions.detect_potential_sensitive_attributes(df_inspect, target_col=target_col_name)
print(f"Detected Attributes for Analysis (re-initialized): {suggestions}")

print("--- Automated Bias Suggestion (XGBoost) ---")
suggestions_df = get_bias_suggestions(X_test_wb, y_pred_xgb_wb, suggestions)
display(suggestions_df)

top_suggestion = suggestions_df.iloc[0]['Attribute']
print(f"\nAI Suggestion: '{top_suggestion}' shows the highest bias. You should likely evaluate this first.")

# 2. Interactive Selection based on Suggestions
SELECTED_BIAS_ATTR = (input(f"Based on the suggestions above, enter the attribute you wish to analyze (Default is '{top_suggestion}'): ") or top_suggestion).strip()

print(f"Selected for Deep-Dive: {SELECTED_BIAS_ATTR}")
from sklearn.metrics import accuracy_score

print("\n--- Overall Model Accuracy ---")
accuracy_xgb = accuracy_score(y_test_wb, y_pred_xgb_wb)
accuracy_mlp = accuracy_score(y_test_wb, y_pred_mlp_wb)

print(f"Overall XGBoost Model Accuracy: {accuracy_xgb:.4f}")
print(f"Overall MLP Model Accuracy: {accuracy_mlp:.4f}")

# Hybrid Model Accuracy Calculation
print("\n--- Hybrid Model (XGBoost + MLP) Accuracy ---")
# For a simple hybrid, let's assume equal weighting for now.
# For binary classification with hard predictions (0 or 1), we can sum and round.
# Or, if we had probabilities, we'd average probabilities and then apply a threshold.

# Assuming y_pred_xgb_wb and y_pred_mlp_wb are arrays of 0s and 1s
y_pred_hybrid = (y_pred_xgb_wb + y_pred_mlp_wb) / 2

# For hard predictions, a simple majority vote (rounding to nearest integer)
# In case of a tie (e.g., 0.5), it will round to the closest even number by default if using numpy.around
# For simplicity here, let's just explicitly round up if >= 0.5
y_pred_hybrid_hard = np.where(y_pred_hybrid >= 0.5, 1, 0)

accuracy_hybrid = accuracy_score(y_test_wb, y_pred_hybrid_hard)

print(f"Overall Hybrid (XGBoost + MLP) Model Accuracy: {accuracy_hybrid:.4f}")


from sklearn.ensemble import RandomForestClassifier
print("Training Random Forest Model...")
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_wb, y_train_wb)
y_pred_rf = rf_model.predict(X_test_wb)
accuracy_rf = accuracy_score(y_test_wb, y_pred_rf)
print(f"Overall Random Forest Model Accuracy: {accuracy_rf:.4f}")

import lightgbm as lgb
print("Training LightGBM Model...")
lgbm_model = lgb.LGBMClassifier(random_state=42, verbose=-1)
lgbm_model.fit(X_train_wb, y_train_wb)
y_pred_lgbm = lgbm_model.predict(X_test_wb)
accuracy_lgbm = accuracy_score(y_test_wb, y_pred_lgbm)
print(f"\nOverall LightGBM Model Accuracy: {accuracy_lgbm:.4f}")


# --- 5. Integrate and Run White-Box Branch with Fairness Evaluation ---

# Use the interactive selection from the previous cell
EXTERNAL_SENSITIVE_ATTR = SELECTED_BIAS_ATTR

# Initialize Engine
fairness_engine = FairnessEvaluationEngine(protected_attribute_name=EXTERNAL_SENSITIVE_ATTR)

print(f"\n--- [1] PRIMARY ANALYSIS FOR: {EXTERNAL_SENSITIVE_ATTR} ---")
A_test_wb = X_test_wb[EXTERNAL_SENSITIVE_ATTR]

# Calculate and plot metrics for the selected attribute
metrics_xgb = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_xgb_wb, A_test_wb)
fairness_engine.plot_bias_metrics(metrics_xgb, title=f"XGBoost Bias: {EXTERNAL_SENSITIVE_ATTR}", print_report=True)

metrics_mlp = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_mlp_wb, A_test_wb)
fairness_engine.plot_bias_metrics(metrics_mlp, title=f"MLP Bias: {EXTERNAL_SENSITIVE_ATTR}", print_report=True)

# --- 6. Explainability with SHAP (White-Box XGBoost Example) ---
print("\n========================================")
print("        SHAP EXPLAINABILITY           ")
print("========================================")

# Use the XGBoost model and test data from the white-box branch
# xgb_model_wb is the trained model returned by run_white_box_branch
# X_train_wb is the training dataset for the white-box branch, which is often used for SHAP explainer background data
explainer = shap.TreeExplainer(xgb_model_wb)
shap_values_xgb_wb = explainer.shap_values(X_test_wb) # Renamed variable

print("Generating SHAP summary plot (bar)...")
shap.summary_plot(shap_values_xgb_wb, X_test_wb, plot_type="bar", show=False)
plt.title("SHAP Feature Importance for White-Box XGBoost Model (Bar Plot)")
plt.tight_layout()
plt.show()

print("Generating SHAP summary plot (dot)....")
shap.summary_plot(shap_values_xgb_wb, X_test_wb, show=False)
plt.title("SHAP Summary Plot for White-Box XGBoost Model (Dot Plot)")
plt.tight_layout()
plt.show()

print("SHAP analysis complete. These plots indicate the average impact of each feature on the model's output magnitude and how individual predictions are influenced.")



## Resources

**Note on PDF Conversion and Embedded Images:**

Previously, this notebook was configured to convert the generated markdown report into a PDF. However, in response to a user request, the plots within the markdown report are now embedded directly as base64 encoded images to display them in-place within the Colab output. This change, while ensuring immediate visibility of plots within the report sections, can make direct `pandoc` PDF conversion more challenging or result in very large PDF file sizes, as it attempts to embed all image data. If PDF conversion is desired, consider reverting to saving plots as external files and referencing them, rather than embedding them directly in the markdown.

### **Understanding Fairness Metrics and Key Findings**

To ensure our AI recruitment tool is ethical and compliant, we used several statistical metrics to measure bias across different groups. Here is a breakdown of what these metrics do and what they revealed about your model.

#### **1. Key Fairness Metrics Explained**
*   **Demographic Parity Difference (DPD):** Measures the absolute difference in selection rates between the most favored group and the least favored group. An ideal score is **0.0**.
*   **Disparate Impact (DI):** The ratio of the selection rate of the unprivileged group to the privileged group. The industry standard follows the **'80% Rule'** (DI > 0.8 is acceptable).
*   **Equal Opportunity Difference (EOD):** Measures the difference in **True Positive Rates** (how many qualified candidates were correctly selected) between groups. It ensures that the model is equally 'accurate' for everyone.
*   **Statistical Parity Difference (SPD):** Specifically measures the gap in probability of selection between two groups. A negative value usually indicates bias against the unprivileged group.
*   **Counterfactual Discrimination Rate:** A 'what-if' metric. It measures how often the model's decision changes if we only change the sensitive attribute (e.g., flipping Gender from Male to Female) while keeping everything else constant.

#### **2. Summary of Findings**
Based on the automated audit, here are the critical insights:

*   **Confirmed Statistical Bias:** The **Chi-Square test (p < 0.05)** confirmed that the disparity in 'Gender' is statistically significant and not due to random chance.
*   **Primary Bias Attributes:** **Gender** and **Education Level** were identified as the most biased features, with the highest Combined Bias Scores (approx. 0.08).
*   **Bias Mitigation Trend:** Interestingly, the audit showed that as we move from the raw dataset to the final system output (**Black-Box Surrogate**), the bias actually **decreased**. For example, Gender DPD dropped from **0.116 (Dataset)** to **0.035 (System)**, suggesting that existing system rules might be helping to soften the raw data's bias.
*   **Low Counterfactual Risk:** The model passed the individual counterfactual test, and the overall **Counterfactual Discrimination Rate was low (1.20%)**, meaning the model rarely changes its mind based solely on a label flip.
*   **Overall Audit Status:** The **Final Fairness Severity Score was 0.85/100**, placing the system in the **'NEGLIGIBLE BIAS'** category. This indicates that while minor disparities exist, the system is currently low-risk for systemic discrimination.

###  Statistical Validation of Fairness Metrics on Raw Dataset
This section validates the presence of pre-existing human prejudice in the training data using statistical tests on the Ground Truth labels.
To ensure that the observed disparities are not due to random chance, we implement:
1. **Chi-Square Test**: To check if the selection rate is independent of the sensitive attribute.
2. **Bootstrap Confidence Intervals**: To estimate the range of the Statistical Parity Difference.
3. **P-Values**: To determine statistical significance.

In [ ]:
from scipy.stats import chi2_contingency
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns

def perform_direct_dataset_audit(df, attr_name, target_col):
    # print(f"--- Direct Audit: Inherent Bias in {attr_name} ---")

    # 1. Chi-Square Test for Independence
    # Handle potential NaN values by dropping them for contingency table creation
    contingency_table = pd.crosstab(df[attr_name], df[target_col], dropna=True)

    if contingency_table.empty or min(contingency_table.shape) < 2:
        # print(f"Skipping Chi-Square test for {attr_name}: insufficient data or groups after dropping NaNs.")
        chi2, p, dof, ex = np.nan, np.nan, np.nan, np.nan
    else:
        chi2, p, dof, ex = chi2_contingency(contingency_table)

    # print(f"Chi-Square Statistic: {chi2:.4f}")
    # print(f"P-Value: {p:.4e}")
    status = 'Significant' if p < 0.05 else 'Not Significant'
    # print(f"Statistical Significance: {status} (p < 0.05)")

    # 2. Bootstrap Confidence Intervals for Historical SPD
    def calc_historical_spd(sample_df):
        # Ensure there are at least two unique groups to calculate SPD
        if sample_df[attr_name].nunique() < 2:
            return 0.0 # No disparity if only one group

        rates = sample_df.groupby(attr_name)[target_col].mean()
        # Ensure there are at least two rates to compare
        if len(rates) < 2:
            return 0.0
        return rates.max() - rates.min()

    n_iterations = 1000
    boot_stats = []

    # Filter out NaNs for bootstrap sampling to avoid errors
    df_filtered = df.dropna(subset=[attr_name, target_col])
    if len(df_filtered) == 0:
        # print(f"Skipping Bootstrap CI for {attr_name}: no valid data after dropping NaNs.")
        lower, upper, mean_spd = np.nan, np.nan, np.nan
    else:
        for _ in range(n_iterations):
            resampled_df = resample(df_filtered)
            boot_stats.append(calc_historical_spd(resampled_df))

        lower = np.percentile(boot_stats, 2.5) if boot_stats else np.nan
        upper = np.percentile(boot_stats, 97.5) if boot_stats else np.nan
        mean_spd = np.mean(boot_stats) if boot_stats else np.nan

    # print(f"95% Bootstrap CI for Inherent SPD: [{lower:.4f}, {upper:.4f}]")
    # print(f"Mean Inherent SPD: {mean_spd:.4f}")

    return {
        'attribute': attr_name,
        'chi2': chi2,
        'p_value': p,
        'significance_status': status,
        'boot_ci_lower': lower,
        'boot_ci_upper': upper,
        'mean_spd': mean_spd
    }

all_audit_results = []
for attr in suggestions:
    result = perform_direct_dataset_audit(df_inspect, attr, target_col_name)
    all_audit_results.append(result)

# Optional: display collected results to confirm
display(pd.DataFrame(all_audit_results))

In [ ]:
print('--- Visualizing Direct Raw Dataset Bias Audit Results ---')

# Convert results to DataFrame for easier plotting
audit_results_df = pd.DataFrame(all_audit_results)

# Calculate error bar values (distance from mean to CI bounds)
audit_results_df['yerr_lower'] = audit_results_df['mean_spd'] - audit_results_df['boot_ci_lower']
audit_results_df['yerr_upper'] = audit_results_df['boot_ci_upper'] - audit_results_df['mean_spd']

# Filter out rows where mean_spd is NaN (e.g., if bootstrap failed for an attribute)
audit_results_df_cleaned = audit_results_df.dropna(subset=['mean_spd']).reset_index(drop=True) # Reset index to align with bar positions

# Display the full statistical validation results in a table
print("\nDetailed Statistical Validation Results:")
display(audit_results_df_cleaned[['attribute', 'chi2', 'p_value', 'significance_status', 'mean_spd', 'boot_ci_lower', 'boot_ci_upper']].round(4))

plt.figure(figsize=(12, 7))
ax = sns.barplot(
    x='attribute',
    y='mean_spd',
    data=audit_results_df_cleaned,
    palette='viridis'
    # yerr parameter removed from here, will be added manually
)

# Manually add error bars for each bar
for i, row in audit_results_df_cleaned.iterrows():
    # Matplotlib errorbar expects 'yerr' to be (lower_error, upper_error) for each point.
    # For a single point, this should be [[lower_error], [upper_error]].
    ax.errorbar(
        x=i, # Position of the bar on the x-axis
        y=row['mean_spd'],
        yerr=[[row['yerr_lower']], [row['yerr_upper']]], # Asymmetric errors for this single point
        fmt='none', # Do not plot a line for the errorbar
        c='black',  # Color of error bars
        capsize=5,  # Caps on error bars
        elinewidth=1 # Line width of error bars
    )

plt.title('Mean Inherent Statistical Parity Difference (SPD) with 95% CI')
plt.xlabel('Sensitive Attribute')
plt.ylabel('Mean Inherent SPD (Max-Min Selection Rate)')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Add significance annotations
for index, row in audit_results_df_cleaned.iterrows():
    # Determine the upper bound of the error bar for correct annotation positioning
    y_upper_bound = row['mean_spd'] + row['yerr_upper'] if pd.notna(row['yerr_upper']) else row['mean_spd']
    y_offset = y_upper_bound + 0.01 # Position annotation slightly above the error bar

    ax.text(
        index, # x-position is the index of the bar
        y_offset,
        f"Chi2: {row['chi2']:.2f}\nP: {row['p_value']:.2e}\n({row['significance_status']})",
        color='black',
        ha='center',
        va='bottom',
        fontsize=9,
        weight='bold' if row['significance_status'] == 'Significant' else 'normal'
    )

plt.tight_layout()
plt.show()

print('--- Direct Raw Dataset Bias Audit Visualization Complete ---')

#SHAP Explainability for White-Box MLP Model

In [ ]:
# --- 6.1 SHAP Explainability for White-Box MLP Model ---
print("\n========================================")
print("       SHAP MLP EXPLAINABILITY         ")
print("========================================")

# Use the already processed data used for MLP training
# X_train_mlp_processed and X_test_mlp_processed are returned by run_white_box_branch

# Using a small background sample for KernelExplainer to speed up calculation
background = shap.sample(X_train_mlp_processed, 10)
explainer_mlp = shap.KernelExplainer(mlp_model_wb.predict, background)

# To reduce computation time, we will calculate SHAP values for a sample of the test set.
# You can adjust `n_shap_samples` to control the number of samples to explain.
n_shap_samples = 100 # Reduced from full X_test_mlp_processed (approx 24k samples) for faster execution
X_test_mlp_sampled = X_test_mlp_processed.sample(n=min(n_shap_samples, len(X_test_mlp_processed)), random_state=42)

shap_values_mlp = explainer_mlp.shap_values(X_test_mlp_sampled)

print("Generating SHAP summary plot for MLP...")
shap.summary_plot(shap_values_mlp, X_test_mlp_sampled, show=False)
plt.title("SHAP Summary Plot for White-Box MLP Model (Sampled)")
plt.tight_layout()
plt.show()

### 7. Intersectionality Analysis

This section performs an intersectionality analysis to identify compounded biases across combinations of sensitive attributes. We will combine the `Gender` and `Education_Level` attributes to see if specific intersectional groups experience disproportionate outcomes.

In [ ]:
print('\n--- Performing Intersectionality Analysis (Gender x Education_Level) ---')

# Assuming 'Gender' is another sensitive attribute present in X_test_wb
# and 'Education_Level' is the SELECTED_BIAS_ATTR (EXTERNAL_SENSITIVE_ATTR)

# Create a combined intersectional attribute
# Ensure both attributes are treated as strings or categories for combinationa
if 'Gender' in X_test_wb.columns:
    X_test_wb['Gender_Education_Level'] = X_test_wb['Gender'].astype(str) + '_' + X_test_wb[EXTERNAL_SENSITIVE_ATTR].astype(str)
    intersectional_attr_series = X_test_wb['Gender_Education_Level']

    # Initialize FairnessEvaluationEngine for the intersectional attribute
    intersectional_engine = FairnessEvaluationEngine(protected_attribute_name='Gender_Education_Level')

    print('\n--- Fairness Metrics for Intersectional Attribute (XGBoost) ---')
    metrics_xgb_intersectional = intersectional_engine.calculate_all_metrics(y_test_wb, y_pred_xgb_wb, intersectional_attr_series)
    intersectional_engine.plot_bias_metrics(metrics_xgb_intersectional, title=f"XGBoost Intersectional Bias: Gender x {EXTERNAL_SENSITIVE_ATTR}", print_report=True)

    print('\n--- Fairness Metrics for Intersectional Attribute (MLP) ---')
    metrics_mlp_intersectional = intersectional_engine.calculate_all_metrics(y_test_wb, y_pred_mlp_wb, intersectional_attr_series)
    intersectional_engine.plot_bias_metrics(metrics_mlp_intersectional, title=f"MLP Intersectional Bias: Gender x {EXTERNAL_SENSITIVE_ATTR}", print_report=True)
else:
    print("Skipping intersectionality analysis: 'Gender' column not found in X_test_wb.")


### Interpretation and Mitigation Strategies for Severe Intersectional Bias

When `Demographic Parity Difference`, `Equal Opportunity Difference`, and `Equalized Odds Difference` are all `1.0000` for certain intersectional groups, it signifies a complete failure of fairness, indicating that one group is always selected (or rejected) while another is always rejected (or selected).

**Interpretation:**
*   **Perfect Disparity:** This is the most extreme form of bias, where the model's decisions are perfectly correlated with the sensitive attribute combination in a discriminatory way. For example, all 'Female_High School' candidates might be rejected, while all 'Male_PhD' candidates are selected, regardless of other qualifications.
*   **Data Imbalance/Feature Omission:** This severe disparity often stems from extreme imbalances in the training data for these specific intersectional groups, or from the model effectively ignoring other relevant features for these groups due to strong signals from the sensitive attributes.

**High-Level Mitigation Strategies:**

1.  **Targeted Data Augmentation/Re-sampling:** Increase the representation of the severely disadvantaged groups in the training data, possibly by oversampling them or generating synthetic data points.
2.  **Re-weighting of Training Samples:** Assign higher weights to samples from disadvantaged groups during model training to ensure their impact on the loss function is amplified.
3.  **Algorithmic Debiasing:** Employ in-processing techniques that modify the learning algorithm itself to be fairer, such as adversarial debiasing or regularizing for fairness constraints.
4.  **Post-processing Interventions:** Adjust model predictions after they are made to enforce fairness. For example, setting minimum acceptance rates for severely disadvantaged groups, even if it slightly reduces overall accuracy.
5.  **Feature Re-evaluation:** Investigate if the model is over-relying on proxies for these sensitive attributes, or if there are missing non-discriminatory features that could better inform decisions for these groups.

Addressing this severe intersectional bias is paramount for establishing a truly fair recruitment system.

### 8. Error Disparity Visualization

This section visualizes the types of errors (True Positives, False Positives, True Negatives, False Negatives) made by the models for different groups within the selected sensitive attribute (`Education_Level`). This provides a more granular view of how misclassifications disproportionately affect various groups.

In [ ]:
print('\n--- Visualizing Error Disparity for selected attribute ---')

def plot_error_disparity(y_true, y_pred, sensitive_attr_series, model_name, positive_label=1):
    df_temp = pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'sensitive_attr': sensitive_attr_series})
    groups = sensitive_attr_series.unique()

    all_error_data = []
    for group in groups:
        group_df = df_temp[df_temp['sensitive_attr'] == group]
        if len(group_df) == 0: continue

        tp = ((group_df['y_true'] == positive_label) & (group_df['y_pred'] == positive_label)).sum()
        fp = ((group_df['y_true'] != positive_label) & (group_df['y_pred'] == positive_label)).sum()
        tn = ((group_df['y_true'] != positive_label) & (group_df['y_pred'] != positive_label)).sum()
        fn = ((group_df['y_true'] == positive_label) & (group_df['y_pred'] != positive_label)).sum()

        total_group = len(group_df)
        all_error_data.append({
            'Group': group,
            'Type': 'True Positive', 'Count': tp, 'Rate': tp/total_group
        })
        all_error_data.append({
            'Group': group,
            'Type': 'False Positive', 'Count': fp, 'Rate': fp/total_group
        })
        all_error_data.append({
            'Group': group,
            'Type': 'True Negative', 'Count': tn, 'Rate': tn/total_group
        })
        all_error_data.append({
            'Group': group,
            'Type': 'False Negative', 'Count': fn, 'Rate': fn/total_group
        })

    error_df = pd.DataFrame(all_error_data)

    plt.figure(figsize=(12, 7))
    sns.barplot(x='Group', y='Rate', hue='Type', data=error_df, palette='muted')
    plt.title(f'Error Disparity for {model_name} by {EXTERNAL_SENSITIVE_ATTR}')
    plt.ylabel('Rate within Group')
    plt.xlabel(EXTERNAL_SENSITIVE_ATTR)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Plot for XGBoost Model
plot_error_disparity(y_test_wb, y_pred_xgb_wb, X_test_wb[EXTERNAL_SENSITIVE_ATTR], 'XGBoost')

# Plot for MLP Model
plot_error_disparity(y_test_wb, y_pred_mlp_wb, X_test_wb[EXTERNAL_SENSITIVE_ATTR], 'MLP')


### Fairness Evaluation Results for Additional Models

Below are the Statistical Parity Difference (SPD), Disparate Impact (DI), and Equal Opportunity Difference (EOD) for the Hybrid, Random Forest, and LightGBM models, using the previously selected sensitive attribute (`EXTERNAL_SENSITIVE_ATTR`).

In [ ]:
fairness_engine = FairnessEvaluationEngine(protected_attribute_name=EXTERNAL_SENSITIVE_ATTR)
A_test_wb = X_test_wb[EXTERNAL_SENSITIVE_ATTR]

# Calculate metrics for Hybrid Model
metrics_hybrid = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_hybrid_hard, A_test_wb)

# Calculate metrics for Random Forest Model
metrics_rf = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_rf, A_test_wb)

# Calculate metrics for LightGBM Model
metrics_lgbm = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_lgbm, A_test_wb)

# Compile results into a DataFrame for display
results_data = {
    'Model': ['XGBoost', 'MLP Neural Network', 'Hybrid Model', 'Random Forest', 'LightGBM'],
    'Statistical Parity Difference (SPD)': [
        metrics_xgb['statistical_parity_difference'],
        metrics_mlp['statistical_parity_difference'],
        metrics_hybrid['statistical_parity_difference'],
        metrics_rf['statistical_parity_difference'],
        metrics_lgbm['statistical_parity_difference']
    ],
    'Disparate Impact (DI)': [
        metrics_xgb['disparate_impact'],
        metrics_mlp['disparate_impact'],
        metrics_hybrid['disparate_impact'],
        metrics_rf['disparate_impact'],
        metrics_lgbm['disparate_impact']
    ],
    'Equal Opportunity Difference (EOD)': [
        metrics_xgb['equal_opportunity_difference'],
        metrics_mlp['equal_opportunity_difference'],
        metrics_hybrid['equal_opportunity_difference'],
        metrics_rf['equal_opportunity_difference'],
        metrics_lgbm['equal_opportunity_difference']
    ]
}

fairness_results_df = pd.DataFrame(results_data)

print(f"Table 3. Fairness Evaluation Results for '{EXTERNAL_SENSITIVE_ATTR}'")
display(fairness_results_df.round(6))

## Section 1: Dataset Bias Detection


### 1. Representation Bias Detection

This section analyzes the distribution of different groups within identified sensitive attributes to detect representation biases. We will compute group distribution percentages, imbalance ratios, and visualize these using bar plots. Finally, attributes will be ranked based on their imbalance severity.


In [ ]:
print('--- Performing Representation Bias Detection ---')

representation_bias_data = []

# Using the 'df_inspect' (original loaded dataset) for dataset-level analysis
# And 'suggestions' (list of potential sensitive attributes) from previous cells

for attr in suggestions:
    if attr not in df_inspect.columns:
        print(f"Warning: Attribute '{attr}' not found in the original dataset. Skipping.")
        continue

    print(f"\nAnalyzing representation for: {attr}")

    # Compute group distribution percentages
    # Use value_counts with normalize=True to get percentages
    group_counts = df_inspect[attr].value_counts()
    group_percentages = df_inspect[attr].value_counts(normalize=True) * 100

    # Filter out NaN values if present, as they shouldn't contribute to group distribution in this context
    if df_inspect[attr].isnull().any():
        print(f"Note: Skipping NaN values in '{attr}' for percentage calculation.")
        group_counts = group_counts.dropna()
        group_percentages = group_percentages.dropna()

    print("Group Distribution Percentages:")
    display(group_percentages.to_frame(name='Percentage').round(2))

    # Compute imbalance ratios (Majority Group Count / Minority Group Count)
    # Sort groups by count to easily identify majority and minority
    sorted_counts = group_counts.sort_values(ascending=False)

    if len(sorted_counts) > 1:
        majority_group_count = sorted_counts.iloc[0]
        minority_group_count = sorted_counts.iloc[-1]

        if minority_group_count > 0:
            imbalance_ratio = majority_group_count / minority_group_count
            print(f"Imbalance Ratio (Majority/Minority): {imbalance_ratio:.2f}")
            representation_bias_data.append({
                'Attribute': attr,
                'Imbalance_Ratio': imbalance_ratio,
                'Majority_Group': sorted_counts.index[0],
                'Minority_Group': sorted_counts.index[-1]
            })
        else:
            print("Cannot calculate imbalance ratio: Minority group count is zero.")
    elif len(sorted_counts) == 1:
        print("Only one group present; imbalance ratio not applicable.")
    else:
        print("No valid groups to calculate imbalance ratio.")

    # Visualize representation imbalance (Bar Plot)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=group_percentages.index.astype(str), y=group_percentages.values, palette='viridis')
    plt.title(f'Representation Distribution for {attr}')
    plt.xlabel(attr)
    plt.ylabel('Percentage (%)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Ranking of most imbalanced groups
if representation_bias_data:
    imbalance_df = pd.DataFrame(representation_bias_data)
    imbalance_df = imbalance_df.sort_values(by='Imbalance_Ratio', ascending=False).reset_index(drop=True)
    print("\n--- Ranking of Most Imbalanced Groups (Higher Ratio = More Imbalanced) ---")
    display(imbalance_df)

    print("\nInterpretation of Imbalance Severity:")
    for index, row in imbalance_df.iterrows():
        print(f"  '{row['Attribute']}' has an imbalance ratio of {row['Imbalance_Ratio']:.2f}, with '{row['Majority_Group']}' being the majority group and '{row['Minority_Group']}' the minority. This indicates a significant representational disparity.")
else:
    print("No representation bias data to display or rank.")

print('\n--- Representation Bias Detection Complete ---')


#Performing Historical Bias Detection

In [ ]:
print('--- Performing Historical Bias Detection ---')

historical_bias_results = []
target_col = 'Hiring_Decision'

for attr in suggestions:
    if attr not in df_inspect.columns:
        continue

    print(f"\nHistorical Hiring Analysis for: {attr}")

    # Calculate selection (hiring) rates per group
    stats = df_inspect.groupby(attr)[target_col].agg(['count', 'mean'])
    stats.columns = ['Total_Candidates', 'Hiring_Rate']
    stats['Hiring_Rate_Pct'] = stats['Hiring_Rate'] * 100

    # Identify majority and minority by count for disparity baseline
    majority_group = df_inspect[attr].value_counts().idxmax()
    majority_rate = stats.loc[majority_group, 'Hiring_Rate']

    stats['Statistical_Disparity'] = stats['Hiring_Rate'] - majority_rate

    display(stats.round(4))

    # Visualization of Hiring Rates
    plt.figure(figsize=(10, 5))
    sns.barplot(x=stats.index.astype(str), y=stats['Hiring_Rate_Pct'], palette='magma')
    plt.axhline(majority_rate * 100, color='red', linestyle='--', label=f'Baseline ({majority_group})')
    plt.title(f'Historical Hiring Rates by {attr}')
    plt.ylabel('Hiring Rate (%)')
    plt.legend()
    plt.xticks(rotation=45)
    plt.show()

    # Store summary for ranking
    max_disparity = stats['Statistical_Disparity'].abs().max()
    historical_bias_results.append({
        'Attribute': attr,
        'Max_Disparity': max_disparity,
        'Disadvantaged_Group': stats['Hiring_Rate'].idxmin(),
        'Advantaged_Group': stats['Hiring_Rate'].idxmax()
    })

# Rank attributes by historical bias severity
hist_ranking_df = pd.DataFrame(historical_bias_results).sort_values(by='Max_Disparity', ascending=False)
print("\n--- Ranking of Attributes by Historical Bias Severity ---")
display(hist_ranking_df)

print('\n--- Historical Bias Detection Complete ---')


### 3. Intersectional Dataset Bias Analysis

This final part of Section 1 explores compounded biases by looking at the interaction between sensitive attributes. We will create a heatmap to visualize the hiring rates for different combinations of `Gender` and `Education_Level` to identify if specific intersectional groups are severely disadvantaged.

In [ ]:
print('--- Performing Intersectional Dataset Bias Analysis ---')

# Grouping by multiple attributes to find intersectional hiring rates
intersectional_stats = df_inspect.groupby(['Gender', 'Education_Level'])[target_col].mean().unstack()

print("\nIntersectional Hiring Rates (Gender x Education Level):")
display(intersectional_stats.round(4))

# Visualization via Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(intersectional_stats, annot=True, fmt='.2%', cmap='YlOrRd', cbar_kws={'label': 'Hiring Rate'})
plt.title('Intersectional Hiring Rates: Gender vs Education Level')
plt.xlabel('Education Level')
plt.ylabel('Gender')
plt.show()

# Identifying the most disadvantaged intersectional group
min_rate = intersectional_stats.min().min()
min_idx = intersectional_stats.stack().idxmin()

print(f"\nObservation: The group with the lowest historical hiring rate is {min_idx} at {min_rate:.2%}.")
print('\n--- Section 1: Dataset Bias Detection Complete ---')


In [ ]:
import pandas as pd
import numpy as np

# Assuming 'all_attributes_comparison_data' and 'suggestions' are available from previous execution

# Define the metric labels in the order they appear in the comparison dataframes
metric_labels = [
    'Demographic Parity Difference',
    'Statistical Parity Difference',
    'Selection Rate Difference',
    'Disparate Impact',
    'Equal Opportunity Difference',
    'FPR Difference',
    'FNR Difference',
    'Average Odds Difference'
]

# Initialize the dictionary to store comparison data for all attributes
all_attributes_comparison_data = {}

# Instantiate the FairnessEvaluationEngine
# This assumes `FairnessEvaluationEngine` class is defined and available from previous cells (e.e.g., SljAEprdgpk6).
fairness_engine = FairnessEvaluationEngine()

print("\n--- Calculating Comprehensive Bias Transformation Data ---")

for attr in suggestions:
    metrics_dict = {
        'Inherent Dataset Bias': {},
        'White-Box XGBoost': {},
        'White-Box MLP': {},
        'Black-Box Surrogate': {}
    }

    # 1. Inherent Dataset Bias (using y_test_wb as both true and predicted to get dataset disparity)
    try:
        # Ensure X_test_wb[attr] is aligned with y_test_wb
        # We can use the fairness_engine's calculate_all_metrics function.
        # To get inherent bias, we pass y_test_wb as both y_true and y_pred.
        dataset_metrics = fairness_engine.calculate_all_metrics(y_test_wb, y_test_wb, X_test_wb[attr])
        metrics_dict['Inherent Dataset Bias'] = {
            'Demographic Parity Difference': dataset_metrics['demographic_parity_difference'],
            'Statistical Parity Difference': dataset_metrics['statistical_parity_difference'],
            'Selection Rate Difference': dataset_metrics['selection_rate_difference'],
            'Disparate Impact': dataset_metrics['disparate_impact'],
            'Equal Opportunity Difference': dataset_metrics['equal_opportunity_difference'],
            'FPR Difference': dataset_metrics['false_positive_rate_difference'],
            'FNR Difference': dataset_metrics['false_negative_rate_difference'],
            'Average Odds Difference': dataset_metrics['average_odds_difference']
        }
    except Exception as e:
        print(f"Warning: Could not calculate Inherent Dataset Bias for '{attr}': {e}")

    # 2. White-Box XGBoost
    try:
        xgb_metrics = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_xgb_wb, X_test_wb[attr])
        metrics_dict['White-Box XGBoost'] = {
            'Demographic Parity Difference': xgb_metrics['demographic_parity_difference'],
            'Statistical Parity Difference': xgb_metrics['statistical_parity_difference'],
            'Selection Rate Difference': xgb_metrics['selection_rate_difference'],
            'Disparate Impact': xgb_metrics['disparate_impact'],
            'Equal Opportunity Difference': xgb_metrics['equal_opportunity_difference'],
            'FPR Difference': xgb_metrics['false_positive_rate_difference'],
            'FNR Difference': xgb_metrics['false_negative_rate_difference'],
            'Average Odds Difference': xgb_metrics['average_odds_difference']
        }
    except Exception as e:
        print(f"Warning: Could not calculate White-Box XGBoost metrics for '{attr}': {e}")

    # 3. White-Box MLP
    try:
        mlp_metrics = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_mlp_wb, X_test_wb[attr])
        metrics_dict['White-Box MLP'] = {
            'Demographic Parity Difference': mlp_metrics['demographic_parity_difference'],
            'Statistical Parity Difference': mlp_metrics['statistical_parity_difference'],
            'Selection Rate Difference': mlp_metrics['selection_rate_difference'],
            'Disparate Impact': mlp_metrics['disparate_impact'],
            'Equal Opportunity Difference': mlp_metrics['equal_opportunity_difference'],
            'FPR Difference': mlp_metrics['false_positive_rate_difference'],
            'FNR Difference': mlp_metrics['false_negative_rate_difference'],
            'Average Odds Difference': mlp_metrics['average_odds_difference']
        }
    except Exception as e:
        print(f"Warning: Could not calculate White-Box MLP metrics for '{attr}': {e}")

    # 4. Black-Box Surrogate (using y_surrogate, y_pred_full_surrogate from QaLnFnLniCvD)
    try:
        # Ensure blackbox_df[attr] is aligned with y_surrogate and y_pred_full_surrogate
        surrogate_metrics = calculate_all_metrics(y_surrogate, y_pred_full_surrogate, blackbox_df[attr]) # Use the calculate_all_metrics from the blackbox section for consistency
        metrics_dict['Black-Box Surrogate'] = {
            'Demographic Parity Difference': surrogate_metrics['demographic_parity_difference'],
            'Statistical Parity Difference': surrogate_metrics['statistical_parity_difference'],
            'Selection Rate Difference': surrogate_metrics['selection_rate_difference'],
            'Disparate Impact': surrogate_metrics['disparate_impact'],
            'Equal Opportunity Difference': surrogate_metrics['equal_opportunity_difference'],
            'FPR Difference': surrogate_metrics['false_positive_rate_difference'],
            'FNR Difference': surrogate_metrics['false_negative_rate_difference'],
            'Average Odds Difference': surrogate_metrics['average_odds_difference']
        }
    except Exception as e:
        print(f"Warning: Could not calculate Black-Box Surrogate metrics for '{attr}': {e}")

    # Create DataFrame for current attribute's comparison
    comparison_df_current_attr = pd.DataFrame(metrics_dict)
    all_attributes_comparison_data[attr] = comparison_df_current_attr
    print(f"Calculated comparison data for attribute: '{attr}'")

print("--- Comprehensive Bias Transformation Data Calculation Complete ---")

# Identify attributes that were already displayed in the previous truncated output
processed_attributes = ['Gender', 'Education_Level'] # Based on the previous standard_output
remaining_attributes = [attr for attr in suggestions if attr not in processed_attributes]

print("\n--- Continuing Comprehensive Bias Transformation Analysis for Remaining Sensitive Attributes ---")

for current_attr in remaining_attributes:
    if current_attr in all_attributes_comparison_data:
        comparison_df_current_attr = all_attributes_comparison_data[current_attr].copy() # Use .copy() to avoid SettingWithCopyWarning

        # Explicitly set the index labels before printing
        comparison_df_current_attr.index = metric_labels

        print(f"\n### Fairness Metric Comparison for '{current_attr}':")
        # Use to_markdown with index=True to ensure full table output with metric names
        print(comparison_df_current_attr.round(4).to_markdown(index=True))

    else:
        print(f"No data found for attribute '{current_attr}' in all_attributes_comparison_data.")

print("### Fairness Metric Comparison for 'Gender':")
# Remove the 'Inherent Dataset Bias' column if it exists
comparison_df_gender = all_attributes_comparison_data['Gender'].copy()
# Explicitly set the index labels before printing
comparison_df_gender.index = metric_labels
print(comparison_df_gender.round(4).to_markdown(index=True))

print("### Fairness Metric Comparison for 'Education_Level':")
# Remove the 'Inherent Dataset Bias' column if it exists
comparison_df_education = all_attributes_comparison_data['Education_Level'].copy()
# Explicitly set the index labels before printing
comparison_df_education.index = metric_labels
print(comparison_df_education.round(4).to_markdown(index=True))
print("\n--- Comprehensive Bias Transformation Analysis Complete ---")

#WHITE-BOX FAIRNESS COMPARISON TABLE

In [ ]:
# ============================================================
# PAPER-READY WHITE-BOX FAIRNESS COMPARISON TABLE
# ============================================================

print("\n" + "="*100)
print("WHITE-BOX FAIRNESS COMPARISON TABLE")
print("="*100)

# ============================================================
# STEP 1: DATASET BASELINE FAIRNESS
# ============================================================

dataset_dpd = abs(
    all_attributes_comparison_data[EXTERNAL_SENSITIVE_ATTR].loc[
        "Demographic Parity Difference",
        "Inherent Dataset Bias"
    ]
)

dataset_spd = (
    all_attributes_comparison_data[EXTERNAL_SENSITIVE_ATTR].loc[
        "Statistical Parity Difference",
        "Inherent Dataset Bias"
    ]
)

dataset_di = (
    all_attributes_comparison_data[EXTERNAL_SENSITIVE_ATTR].loc[
        "Disparate Impact",
        "Inherent Dataset Bias"
    ]
)

dataset_eod = (
    all_attributes_comparison_data[EXTERNAL_SENSITIVE_ATTR].loc[
        "Equal Opportunity Difference",
        "Inherent Dataset Bias"
    ]
)

print(f"\nDataset Baseline DPD : {dataset_dpd:.4f}")
print(f"Dataset Baseline SPD : {dataset_spd:.4f}")
print(f"Dataset Baseline DI  : {dataset_di:.4f}")
print(f"Dataset Baseline EOD : {dataset_eod:.4f}")

# ============================================================
# STEP 2: STORE MODEL INFORMATION
# ============================================================

model_info = [
    ("XGBoost", metrics_xgb, accuracy_xgb),
    ("MLP", metrics_mlp, accuracy_mlp),
    ("Random Forest", metrics_rf, accuracy_rf),
    ("LightGBM", metrics_lgbm, accuracy_lgbm),
    ("Hybrid", metrics_hybrid, accuracy_hybrid)
]

# ============================================================
# STEP 3: CREATE DATASET ROW
# ============================================================

comparison_rows = []

comparison_rows.append({

    "Model": "Dataset",

    "Accuracy": np.nan,

    "SPD": dataset_spd,

    "DPD": dataset_dpd,

    "DI": dataset_di,

    "EOD": dataset_eod,

    "BAF (%)": 0.0,

    "Fairness Status": "Historical Bias"

})

# ============================================================
# STEP 4: PROCESS EACH MODEL
# ============================================================

for model_name, metrics, accuracy in model_info:

    spd = metrics["statistical_parity_difference"]

    dpd = abs(
        metrics["demographic_parity_difference"]
    )

    di = metrics["disparate_impact"]

    eod = metrics["equal_opportunity_difference"]

    # --------------------------------------------------------
    # BIAS AMPLIFICATION FACTOR
    # --------------------------------------------------------

    if dataset_dpd > 0:

        baf = (
            (dpd - dataset_dpd)
            / dataset_dpd
        ) * 100

    else:

        baf = 0

    # --------------------------------------------------------
    # FAIRNESS STATUS
    # --------------------------------------------------------

    if baf < 0:

        status = "Bias Mitigation"

    elif baf > 0:

        status = "Bias Amplification"

    else:

        status = "No Change"

    comparison_rows.append({

        "Model": model_name,

        "Accuracy": accuracy,

        "SPD": spd,

        "DPD": dpd,

        "DI": di,

        "EOD": eod,

        "BAF (%)": baf,

        "Fairness Status": status

    })

# ============================================================
# STEP 5: CREATE DATAFRAME
# ============================================================

whitebox_comparison_df = pd.DataFrame(
    comparison_rows
)

# ============================================================
# STEP 6: IDENTIFY BEST FAIR MODEL
# Lowest DPD = Best Fairness
# ============================================================

model_rows = whitebox_comparison_df[
    whitebox_comparison_df["Model"] != "Dataset"
]

best_model_index = model_rows[
    "DPD"
].idxmin()

whitebox_comparison_df.loc[
    best_model_index,
    "Fairness Status"
] = "Best Fairness Performance"

# ============================================================
# STEP 7: ROUND NUMERIC VALUES
# ============================================================

numeric_cols = [
    "Accuracy",
    "SPD",
    "DPD",
    "DI",
    "EOD",
    "BAF (%)"
]

whitebox_comparison_df[
    numeric_cols
] = (
    whitebox_comparison_df[
        numeric_cols
    ].round(4)
)

# ============================================================
# STEP 8: DISPLAY TABLE
# ============================================================

print("\nFINAL WHITE-BOX FAIRNESS COMPARISON TABLE\n")

display(
    whitebox_comparison_df
)

# ============================================================
# STEP 9: PRINT SUMMARY
# ============================================================

best_model_name = (
    whitebox_comparison_df.loc[
        best_model_index,
        "Model"
    ]
)

best_model_dpd = (
    whitebox_comparison_df.loc[
        best_model_index,
        "DPD"
    ]
)

print("\n" + "="*100)
print(f"BEST FAIR MODEL : {best_model_name}")
print(f"LOWEST DPD      : {best_model_dpd:.4f}")
print("="*100)

### Section: Advanced Performance & Bias Amplification in white box
This cell calculates the missing ROC-AUC metrics for white-box models and implements the framework's specific **Bias Amplification** formula: $DPD_{model} - DPD_{dataset}$.

In [ ]:
from sklearn.metrics import roc_auc_score

# Prepare features for XGBoost ensuring categorical types and correct column order
X_test_xgb = X_test_wb.copy()
# Standardize categorical columns
for col in X_test_xgb.select_dtypes(['object', 'category']).columns:
    X_test_xgb[col] = X_test_xgb[col].astype('category')

# Ensure feature names match model training precisely
model_features = xgb_model_wb.get_booster().feature_names
X_test_xgb = X_test_xgb[model_features]

# 1. ROC-AUC Calculation
auc_xgb = roc_auc_score(y_test_wb, xgb_model_wb.predict_proba(X_test_xgb)[:, 1])
auc_mlp = roc_auc_score(y_test_wb, mlp_model_wb.predict_proba(X_test_mlp_processed)[:, 1])

print(f"XGBoost ROC-AUC: {auc_xgb:.4f}")
print(f"MLP ROC-AUC: {auc_mlp:.4f}")

# 2. Bias Amplification Tracking
amplification_results = []
for attr in suggestions:
    dpd_dataset = abs(all_attributes_comparison_data[attr].loc['Demographic Parity Difference', 'Inherent Dataset Bias'])
    dpd_model = abs(all_attributes_comparison_data[attr].loc['Demographic Parity Difference', 'White-Box XGBoost'])

    amp_score = dpd_model - dpd_dataset
    amplification_results.append({
        'Attribute': attr,
        'Dataset_DPD': dpd_dataset,
        'Model_DPD': dpd_model,
        'Bias_Amplification': amp_score
    })

amp_df = pd.DataFrame(amplification_results).sort_values(by='Bias_Amplification', ascending=False)
print("\n--- Bias Amplification Ranking (Positive = Model increased bias) ---")
display(amp_df)

#Black box Model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# 1. LOAD DATASET
df = pd.read_csv('/content/fair_recrutment_dataset final.csv')

# Define target and features
X = df.drop('Hiring_Decision', axis=1)
y = df['Hiring_Decision']

# Group columns for preprocessing
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

# 2. PREPROCESSING PIPELINE
# Numerical: Median imputation + Scaling
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: 'missing' tag + OneHot Encoding
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numerical_cols),
        ('cat', cat_transformer, categorical_cols)
    ])

# 3. TRAIN RANDOM FOREST MODEL
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)

# 4. EVALUATION & SELECTION BASIS (FEATURE IMPORTANCE)
print(f"Model Training Complete. Accuracy: {accuracy_score(y_test, model_pipeline.predict(X_test)):.2%}")

# Visualize basis of selection
feature_names = (numerical_cols +
                 list(model_pipeline.named_steps['preprocessor'].named_transformers_['cat']
                      .named_steps['onehot'].get_feature_names_out(categorical_cols)))
importances = model_pipeline.named_steps['classifier'].feature_importances_
indices = np.argsort(importances)[-10:]

plt.figure(figsize=(10, 5))
plt.title('Top 10 Factors Influencing Hiring Selection (Selection Basis)')
plt.barh(range(len(indices)), importances[indices], color='teal')
plt.yticks(range(len(indices)), [feature_names[i] for i in indices])
plt.xlabel('Importance Score')
plt.show()

# 5. PREDICTION INTERFACE
def predict_candidate(candidate_data):
    """
    Accepts a dictionary of candidate info.
    Missing attributes are handled automatically.
    """
    # Create DataFrame from input
    input_df = pd.DataFrame([candidate_data])

    # Align with model features (add missing columns as NaN)
    for col in X.columns:
        if col not in input_df.columns:
            input_df[col] = np.nan

    input_df = input_df[X.columns] # Ensure correct order

    # Generate Prediction
    prob = model_pipeline.predict_proba(input_df)[0][1]
    decision = "SELECTED" if prob >= 0.5 else "REJECTED"

    return {
        "Decision": decision,
        "Selection Probability": f"{prob:.2%}",
        "Status": "Success"
    }

def predict_candidate_v2(candidate_data):
    """
    Enhanced prediction function with manual weights and interaction logic.
    Modified to align with the 15 primary attributes and handle expert rules.
    """
    input_df = pd.DataFrame([candidate_data])
    for col in X.columns:
        if col not in input_df.columns:
            input_df[col] = np.nan
    input_df = input_df[X.columns]

    # 1. Base AI Probability
    base_prob = model_pipeline.predict_proba(input_df)[0][1]
    modified_prob = base_prob

    # 2. Expert Penalty: Employment Gaps
    gap = candidate_data.get('EmploymentGapMonths', 0)
    if pd.notnull(gap) and gap > 12:
        modified_prob *= 0.85 # 15% reduction for long gaps

    # 3. Expert Bonus: High Tech + High Exp (Seniority)
    # Uses 'Technical_Test_Score' and 'Experience_Years' from the 15 attributes
    tech_score = candidate_data.get('Technical_Test_Score', 0)
    exp = candidate_data.get('Experience_Years', 0)
    if pd.notnull(tech_score) and pd.notnull(exp):
        if tech_score > 85 and exp > 10:
            modified_prob = min(1.0, modified_prob * 1.10) # 10% boost

    # 4. Hard Filter: Minimum Coding Standard
    coding_score = candidate_data.get('CodingTestScore', 100)
    if pd.notnull(coding_score) and coding_score < 40:
        decision = "REJECTED (Hard Filter: Coding)"
    else:
        decision = "SELECTED" if modified_prob >= 0.5 else "REJECTED"

    return {
        "Base AI Probability": f"{base_prob:.2%}",
        "Adjusted Probability": f"{modified_prob:.2%}",
        "Final Decision": decision
    }

import random

test_case = {
    'Candidate_ID': random.randint(1, 100000),
    'Age': 30,
    'Gender': 'Female',
    'Experience_Years': 5,
    'Education_Level': 'Masters',
    'Skill_Score': 85,
    'Aptitude_Test_Score': 75,
    'Technical_Test_Score': 90,
    'Communication_Score': 80,
    'Certifications_Count': 2,
    'Previous_Companies': 3,
    'Interview_Score': 92,
    'Location': 'Urban',
    'Job_Role_Applied': 'Software Engineer',
    'Expected_Salary': 120000

}
print(predict_candidate_v2(test_case))

def generate_candidate():
    return {
        "Candidate_ID": random.randint(1, 100000),
        "Gender": random.choice(["Male", "Female", "Other"]),
        "Age": random.randint(21, 60),
        "Education_Level": random.choice(["High School", "Bachelors", "Masters", "PhD"]),
        "Experience_Years": random.randint(0, 30),
        "Skill_Score": random.randint(30, 100),
        "Aptitude_Test_Score": random.randint(40, 100),
        "Technical_Test_Score": random.randint(35, 100),
        "Communication_Score": random.randint(30, 100),
        "Certifications_Count": random.randint(0, 5),
        "Previous_Companies": random.randint(0, 10),
        "Interview_Score": random.randint(0, 100),
        "Location": random.choice(["Urban", "Rural", "Semi-Urban"]),
        "Job_Role_Applied": random.choice(["Software Engineer", "Data Analyst", "ML Engineer", "HR Executive", "Manager"]),
        "Expected_Salary": random.randint(30000, 200000)

    }

observations = []

for i in range(20000):
    candidate = generate_candidate()
    result = predict_candidate_v2(candidate)
    record = candidate.copy()
    record["Base AI Probability"] = result["Base AI Probability"]
    record["Adjusted Probability"] = result["Adjusted Probability"]
    record["Final Decision"] = result["Final Decision"]
    observations.append(record)

blackbox_df = pd.DataFrame(observations)

blackbox_df.to_csv(
    "blackbox_observation_dataset.csv",
    index=False
)

print("BLACK-BOX OBSERVATION DATASET CREATED")
print(blackbox_df.head())

candidate_male = {
        "Candidate_ID": random.randint(1, 100000),
        "Gender":'Male',
        "Age": 25,
        "Education_Level": 'Masters',
        "Experience_Years": 5,
        "Skill_Score":40,
        "Aptitude_Test_Score":55 ,
        "Technical_Test_Score":60 ,
        "Communication_Score":65,
        "Certifications_Count":5 ,
        "Previous_Companies": 2,
        "Interview_Score": 90,
        "Location":'Urban' ,
        "Job_Role_Applied":'Software Engineer' ,
        "Expected_Salary":4000

}

candidate_female = candidate_male.copy()
candidate_female["Gender"] = "Female"

male_result = predict_candidate_v2(candidate_male)
female_result = predict_candidate_v2(candidate_female)

print("\nMale Candidate Result:")
print(male_result)

print("\nFemale Candidate Result:")
print(female_result)

selection_rates = (
    blackbox_df
    .groupby("Gender")["Final Decision"]
    .value_counts(normalize=True)
)

print("\nSelection Rate By Gender:")
print(selection_rates)

# Extract all feature names from the preprocessor
cat_encoder = model_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_features = list(cat_encoder.get_feature_names_out(categorical_cols))
all_feature_names = numerical_cols + cat_features

# Get importance scores from the RandomForest classifier
importances = model_pipeline.named_steps['classifier'].feature_importances_

# Create a mapping of attribute to its total influence
# For categorical features, we sum the importance of all their one-hot encoded levels
influence_map = {}

# Map numerical features directly
for i, col in enumerate(numerical_cols):
    influence_map[col] = importances[i]

# Map and sum categorical features
start_idx = len(numerical_cols)
for col in categorical_cols:
    # Find all one-hot columns belonging to this category
    col_indices = [i for i, name in enumerate(all_feature_names) if name.startswith(col)]
    influence_map[col] = sum(importances[col_indices])

# Convert to DataFrame for a clean display
influence_df = pd.DataFrame(list(influence_map.items()), columns=['Attribute', 'InfluenceValue'])
influence_df = influence_df.sort_values(by='InfluenceValue', ascending=False).reset_index(drop=True)

print("--- Influence Values for All 20 Attributes ---")
display(influence_df)
display(blackbox_df.head())



import pandas as pd

# Load the blackbox observation dataset
blackbox_df = pd.read_csv('blackbox_observation_dataset.csv')

print("Blackbox observation dataset loaded successfully.")
display(blackbox_df.head())

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Prepare the target variable
# Convert 'SELECTED' to 1 and 'REJECTED' to 0
blackbox_df['Final Decision_Numeric'] = blackbox_df['Final Decision'].apply(lambda x: 1 if x == 'SELECTED' else 0)

# Define features (X_surrogate) and target (y_surrogate)
X_surrogate = blackbox_df.drop(['Base AI Probability', 'Adjusted Probability', 'Final Decision', 'Final Decision_Numeric'], axis=1)
y_surrogate = blackbox_df['Final Decision_Numeric']

# Identify numerical and categorical columns for preprocessing
numerical_cols_surrogate = X_surrogate.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols_surrogate = X_surrogate.select_dtypes(include=['object']).columns.tolist()

# Preprocessing for the surrogate model
# Numerical: Median imputation + Scaling
num_transformer_surrogate = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: 'missing' tag + OneHot Encoding
cat_transformer_surrogate = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Create a preprocessor using ColumnTransformer
preprocessor_surrogate = ColumnTransformer(
    transformers=[
        ('num', num_transformer_surrogate, numerical_cols_surrogate),
        ('cat', cat_transformer_surrogate, categorical_cols_surrogate)
    ])

# Split data into training and testing sets
X_train_surrogate, X_test_surrogate, y_train_surrogate, y_test_surrogate = train_test_split(
    X_surrogate, y_surrogate, test_size=0.2, random_state=42, stratify=y_surrogate)

print("Data preprocessing complete and split into training/testing sets.")
print(f"X_train_surrogate shape: {X_train_surrogate.shape}")
print(f"y_train_surrogate shape: {y_train_surrogate.shape}")

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Define the XGBoost Surrogate Model pipeline
surrogate_model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_surrogate),
    ('classifier', XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, use_label_encoder=False, eval_metric='logloss'))
])

# Train the surrogate model
surrogate_model_pipeline.fit(X_train_surrogate, y_train_surrogate)

print("XGBoost Surrogate Model training complete.")

# Make predictions on the test set
y_pred_surrogate = surrogate_model_pipeline.predict(X_test_surrogate)
y_pred_proba_surrogate = surrogate_model_pipeline.predict_proba(X_test_surrogate)[:, 1]

# Evaluate the surrogate model
accuracy = accuracy_score(y_test_surrogate, y_pred_surrogate)
precision = precision_score(y_test_surrogate, y_pred_surrogate)
recall = recall_score(y_test_surrogate, y_pred_surrogate)
f1 = f1_score(y_test_surrogate, y_pred_surrogate)
conf_matrix = confusion_matrix(y_test_surrogate, y_pred_surrogate)

print(f"\nSurrogate Model Performance:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

# Plot Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Rejected', 'Selected'], yticklabels=['Rejected', 'Selected'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Surrogate Model Confusion Matrix')
plt.show()



from sklearn.metrics import recall_score, precision_score, confusion_matrix
import numpy as np

def demographic_parity(y_pred, sensitive_attr):
    """Calculates demographic parity (selection rate difference)."""
    groups = sensitive_attr.unique()
    group_rates = []
    for group in groups:
        idx = (sensitive_attr == group)
        selection_rate = np.mean(y_pred[idx])
        group_rates.append(selection_rate)
    return abs(group_rates[0] - group_rates[1]), groups

def statistical_parity_difference(y_pred, sensitive_attr):
    """Calculates Statistical Parity Difference (Selection Rate Diff)."""
    diff, _ = demographic_parity(y_pred, sensitive_attr)
    return diff

def false_positive_rate_difference(y_true, y_pred, sensitive_attr):
    """Calculates difference in False Positive Rates."""
    groups = sensitive_attr.unique()
    group_fprs = []
    for group in groups:
        idx = (sensitive_attr == group)
        tn, fp, fn, tp = confusion_matrix(y_true[idx], y_pred[idx], labels=[0, 1]).ravel()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        group_fprs.append(fpr)
    return abs(group_fprs[0] - group_fprs[1])

def false_negative_rate_difference(y_true, y_pred, sensitive_attr):
    """Calculates difference in False Negative Rates."""
    groups = sensitive_attr.unique()
    group_fnrs = []
    for group in groups:
        idx = (sensitive_attr == group)
        tn, fp, fn, tp = confusion_matrix(y_true[idx], y_pred[idx], labels=[0, 1]).ravel()
        fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
        group_fnrs.append(fnr)
    return abs(group_fnrs[0] - group_fnrs[1])

def average_odds_difference(y_true, y_pred, sensitive_attr):
    """Calculates average of FPR difference and TPR difference."""
    groups = sensitive_attr.unique()
    diffs = []
    for group in groups:
        idx = (sensitive_attr == group)
        tn, fp, fn, tp = confusion_matrix(y_true[idx], y_pred[idx], labels=[0, 1]).ravel()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
        diffs.append((tpr, fpr))
    tpr_diff = abs(diffs[0][0] - diffs[1][0])
    fpr_diff = abs(diffs[0][1] - diffs[1][1])
    return 0.5 * (fpr_diff + tpr_diff)

def equal_opportunity(y_true, y_pred, sensitive_attr):
    """Calculates equal opportunity (true positive rate difference)."""
    groups = sensitive_attr.unique()
    group_tpr = []
    for group in groups:
        idx = (sensitive_attr == group)
        tpr = recall_score(y_true[idx], y_pred[idx], pos_label=1, zero_division=0)
        group_tpr.append(tpr)
    return abs(group_tpr[0] - group_tpr[1]), groups

def equalized_odds(y_true, y_pred, sensitive_attr):
    groups = sensitive_attr.unique()
    group_tpr, group_fpr = [], []
    for group in groups:
        idx = (sensitive_attr == group)
        tn, fp, fn, tp = confusion_matrix(y_true[idx], y_pred[idx], labels=[0, 1]).ravel()
        group_tpr.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        group_fpr.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    return (abs(group_tpr[0] - group_tpr[1]), abs(group_fpr[0] - group_fpr[1])), groups

def disparate_impact(y_pred, sensitive_attr):
    groups = sensitive_attr.unique()
    group_rates = [np.mean(y_pred[sensitive_attr == g]) for g in groups]
    if min(group_rates) == 0: return np.inf, groups
    return max(group_rates) / min(group_rates), groups

print("Fairness Metric Engine Updated with Additional Metrics.")


def calculate_all_metrics(y_true, y_pred, attr):
    """Helper to return a dictionary of all calculated metrics for an attribute."""
    dp_diff, _ = demographic_parity(y_pred, attr)
    eo_diff, _ = equal_opportunity(y_true, y_pred, attr)
    eodds_diff, _ = equalized_odds(y_true, y_pred, attr)
    di_ratio, _ = disparate_impact(y_pred, attr)

    return {
        'demographic_parity_difference': dp_diff,
        'statistical_parity_difference': statistical_parity_difference(y_pred, attr),
        'selection_rate_difference': dp_diff, # DP and SR are equivalent here
        'equal_opportunity_difference': eo_diff,
        'false_positive_rate_difference': false_positive_rate_difference(y_true, y_pred, attr),
        'false_negative_rate_difference': false_negative_rate_difference(y_true, y_pred, attr),
        'average_odds_difference': average_odds_difference(y_true, y_pred, attr),
        'disparate_impact': di_ratio
    }

def get_composite_bias_score(y_true, y_pred, attr):
    """Calculates a weighted, normalized bias score."""
    metrics = calculate_all_metrics(y_true, y_pred, attr)

    # Define weights as requested
    metric_weights = {
        'demographic_parity_difference': 0.2,
        'disparate_impact': 0.2,
        'equal_opportunity_difference': 0.2,
        'selection_rate_difference': 0.1,
        'false_positive_rate_difference': 0.1,
        'false_negative_rate_difference': 0.1,
        'average_odds_difference': 0.05,
        'statistical_parity_difference': 0.05
    }

    # Normalize weights to ensure sum == 1
    total_weight = sum(metric_weights.values())
    normalized_weights = {k: v / total_weight for k, v in metric_weights.items()}

    composite_bias_score = 0.0

    for metric_name, weight in normalized_weights.items():
        val = metrics[metric_name]

        # Normalization logic:
        # Differences: abs(val) -> 0 is ideal
        # Ratios: abs(1 - val) -> 1 is ideal
        if metric_name == 'disparate_impact':
            normalized_val = abs(1.0 - val)
        else:
            normalized_val = abs(val)

        composite_bias_score += (weight * normalized_val)

    return composite_bias_score, metrics

print("Composite Bias Score Engine Initialized.")

import pandas as pd

# 1. Ensure the AgeGroup column exists for the audit
age_bins = [0, 30, 40, 100]
age_labels = ['Youth', 'Adult', 'Senior']
blackbox_df['AgeGroup'] = pd.cut(blackbox_df['Age'], bins=age_bins, labels=age_labels, right=False)

# 2. FIX: Generate predictions for the entire dataset using the surrogate model
# This defines 'y_pred_full_surrogate' which was causing the NameError
X_full_processed = surrogate_model_pipeline.named_steps['preprocessor'].transform(X_surrogate)
y_pred_full_surrogate = surrogate_model_pipeline.named_steps['classifier'].predict(X_full_processed)

# 3. Perform audit on primary sensitive attributes
attributes_to_audit = {
    "Gender": blackbox_df['Gender'],
    "Age (Youth vs Adult)": blackbox_df[blackbox_df['AgeGroup'].isin(['Youth', 'Adult'])]['AgeGroup']
}

bias_ranking = []

for label, attr_series in attributes_to_audit.items():
    # Align target and predictions for subsets
    current_y_true = y_surrogate[attr_series.index]
    current_y_pred = y_pred_full_surrogate[attr_series.index]

    # Calculate the composite score
    score, _ = get_composite_bias_score(current_y_true, current_y_pred, attr_series)
    bias_ranking.append({"Attribute": label, "Composite Bias Score": score})

# 4. Create and Display Ranking Table
ranking_df = pd.DataFrame(bias_ranking).sort_values(by="Composite Bias Score", ascending=False)

print("--- Final Fairness Ranking (Higher Score = Higher Bias) ---")
display(ranking_df)

# Predict on the entire blackbox dataset using the surrogate model
X_full_processed = surrogate_model_pipeline.named_steps['preprocessor'].transform(X_surrogate)
y_pred_full_surrogate = surrogate_model_pipeline.named_steps['classifier'].predict(X_full_processed)

def run_comprehensive_audit(y_true, y_pred, attr, label):
    print(f"\n--- Comprehensive Fairness Evaluation for {label} ---")
    # Existing metrics
    dp_diff, _ = demographic_parity(y_pred, attr)
    eo_diff, _ = equal_opportunity(y_true, y_pred, attr)
    eodds_diff, _ = equalized_odds(y_true, y_pred, attr)
    di_ratio, _ = disparate_impact(y_pred, attr)

    # New metrics
    fpr_diff = false_positive_rate_difference(y_true, y_pred, attr)
    fnr_diff = false_negative_rate_difference(y_true, y_pred, attr)
    avg_odds = average_odds_difference(y_true, y_pred, attr)
    stat_parity = statistical_parity_difference(y_pred, attr)

    print(f"Demographic Parity Difference: {dp_diff:.4f}")
    print(f"Statistical Parity Difference: {stat_parity:.4f}")
    print(f"Selection Rate Difference: {dp_diff:.4f}")
    print(f"Equal Opportunity (TPR Diff): {eo_diff:.4f}")
    print(f"False Positive Rate Difference: {fpr_diff:.4f}")
    print(f"False Negative Rate Difference: {fnr_diff:.4f}")
    print(f"Average Odds Difference: {avg_odds:.4f}")
    print(f"Equalized Odds (TPR Diff: {eodds_diff[0]:.4f}, FPR Diff: {eodds_diff[1]:.4f})")
    print(f"Disparate Impact Ratio: {di_ratio:.4f}")

# Audit for Gender
run_comprehensive_audit(y_surrogate, y_pred_full_surrogate, blackbox_df['Gender'], "Gender")

# Audit for Age (Youth vs Adult)
age_bins = [0, 30, 40, 100]
age_labels = ['Youth', 'Adult', 'Senior']
blackbox_df['AgeGroup'] = pd.cut(blackbox_df['Age'], bins=age_bins, labels=age_labels, right=False)
sub_df_age = blackbox_df[blackbox_df['AgeGroup'].isin(['Youth', 'Adult'])]
run_comprehensive_audit(y_surrogate[sub_df_age.index], y_pred_full_surrogate[sub_df_age.index], sub_df_age['AgeGroup'], "Age (Youth vs Adult)")

import matplotlib.pyplot as plt
import seaborn as sns

print("\n--- Group-wise Selection Rate Analysis ---")

# --- Selection Rate by Gender ---
print("\nSelection Rate by Gender:")
gender_selection_df = blackbox_df.groupby('Gender')['Final Decision_Numeric'].mean().reset_index()
gender_selection_df['Selection Rate'] = gender_selection_df['Final Decision_Numeric'] * 100
display(gender_selection_df)

# --- Selection Rate by Age Group ---
print("\nSelection Rate by Age Group:")
age_selection_df = blackbox_df.groupby('AgeGroup')['Final Decision_Numeric'].mean().reset_index()
age_selection_df['Selection Rate'] = age_selection_df['Final Decision_Numeric'] * 100
display(age_selection_df)

# --- Acceptance/Rejection Disparity ---
# Using the full dataset and surrogate predictions for disparity analysis
combined_df = blackbox_df.copy()
combined_df['Surrogate_Prediction'] = y_pred_full_surrogate

print("\n--- Acceptance/Rejection Disparity Analysis ---")

def calculate_disparity(df, sensitive_attr_col, outcome_col='Surrogate_Prediction'):
    disparities = {}
    groups = df[sensitive_attr_col].unique()
    for group in groups:
        group_df = df[df[sensitive_attr_col] == group]
        acceptance_rate = group_df[outcome_col].mean()
        rejection_rate = 1 - acceptance_rate
        disparities[group] = {'Acceptance Rate': acceptance_rate, 'Rejection Rate': rejection_rate}
    return disparities

gender_disparities = calculate_disparity(combined_df, 'Gender')
age_group_disparities = calculate_disparity(combined_df, 'AgeGroup')

print("\nGender Disparities:")
for group, rates in gender_disparities.items():
    print(f"  {group}: Acceptance Rate = {rates['Acceptance Rate']:.2%}, Rejection Rate = {rates['Rejection Rate']:.2%}")

print("\nAge Group Disparities:")
for group, rates in age_group_disparities.items():
    print(f"  {group}: Acceptance Rate = {rates['Acceptance Rate']:.2%}, Rejection Rate = {rates['Rejection Rate']:.2%}")

# --- Identify Worst Affected Group ---
# For this, we'll consider the group with the lowest selection rate from our initial analysis

# Worst affected gender
worst_gender = gender_selection_df.loc[gender_selection_df['Selection Rate'].idxmin()]
print(f"\nWorst Affected Gender Group: {worst_gender['Gender']} with a selection rate of {worst_gender['Selection Rate']:.2f}%")

# Worst affected age group
worst_age_group = age_selection_df.loc[age_selection_df['Selection Rate'].idxmin()]
print(f"Worst Affected Age Group: {worst_age_group['AgeGroup']} with a selection rate of {worst_age_group['Selection Rate']:.2f}%")

import shap
import matplotlib.pyplot as plt

print("\n--- SHAP Explainability Analysis ---")

# Get the trained XGBoost classifier from the pipeline
xgb_classifier = surrogate_model_pipeline.named_steps['classifier']

# Get the preprocessed data for SHAP explanation
# We'll use the preprocessed training data for the Explainer background dataset
X_train_processed_surrogate = surrogate_model_pipeline.named_steps['preprocessor'].transform(X_train_surrogate)

# Get feature names after one-hot encoding
feature_names_out = surrogate_model_pipeline.named_steps['preprocessor'].get_feature_names_out()

# Create a SHAP Explainer
explainer = shap.TreeExplainer(xgb_classifier, data=X_train_processed_surrogate)

# Calculate SHAP values for the preprocessed test set
X_test_processed_surrogate = surrogate_model_pipeline.named_steps['preprocessor'].transform(X_test_surrogate)
shap_values = explainer.shap_values(X_test_processed_surrogate)

# SHAP Summary Plot
print("\nGenerating SHAP Summary Plot...")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_processed_surrogate, feature_names=feature_names_out, show=False)
plt.title('SHAP Summary Plot (Feature Importance & Impact)')
plt.tight_layout()
plt.show()

# SHAP Feature Importance Plot (Mean absolute SHAP value)
print("\nGenerating SHAP Feature Importance Plot...")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_processed_surrogate, feature_names=feature_names_out, plot_type="bar", show=False)
plt.title('SHAP Feature Importance (Mean Absolute SHAP Value)')
plt.tight_layout()
plt.show()

# Influential Feature Analysis
# Top 10 most influential features based on mean absolute SHAP value
shap_importance = pd.DataFrame({
    'Feature': feature_names_out,
    'SHAP_Importance': np.abs(shap_values).mean(axis=0)
})
shap_importance = shap_importance.sort_values(by='SHAP_Importance', ascending=False).reset_index(drop=True)

print("\nTop 10 Most Influential Features (SHAP):")
display(shap_importance.head(10))

# Proxy Discrimination Analysis
# We will check if 'Gender' or 'AgeGroup' related features are highly influential
# This is done by observing their presence in the top influential features
print("\nProxy Discrimination Analysis (observing sensitive attributes in top SHAP features):")
sensitive_features_prefixes = ['Gender', 'Age'] # Age is numerical, Gender is one-hot encoded
almost_sensitive_features = ['PreviousRole', 'EducationTier', 'CurrentLocation', 'PreferredCertification'] # Potential proxies

for feature in shap_importance['Feature']:
    if any(feature.startswith(prefix) for prefix in sensitive_features_prefixes):
        print(f"  - Sensitive attribute '{feature}' is a highly influential feature.")
    elif any(feature.startswith(prefix) for prefix in almost_sensitive_features):
        print(f"  - Potential proxy '{feature}' (related to sensitive attributes) is highly influential.")

print("SHAP explainability analysis complete.")


import matplotlib.pyplot as plt
import seaborn as sns

print("\n--- Generating Visualizations ---")

# --- Gender Selection Comparison Chart ---
plt.figure(figsize=(8, 5))
sns.barplot(x='Gender', y='Selection Rate', data=gender_selection_df, palette='viridis')
plt.title('Selection Rate by Gender (Surrogate Model)')
plt.ylabel('Selection Rate (%)')
plt.ylim(0, 100)
plt.show()

# --- Age-Group Disparity Chart ---
plt.figure(figsize=(10, 6))
sns.barplot(x='AgeGroup', y='Selection Rate', data=age_selection_df, palette='magma')
plt.title('Selection Rate by Age Group (Surrogate Model)')
plt.ylabel('Selection Rate (%)')
plt.ylim(0, 100)
plt.show()

# --- Group Outcome Comparison Plots (Detailed Acceptance/Rejection Rates) ---
def plot_group_outcomes(disparities, title):
    df_plot = pd.DataFrame.from_dict(disparities, orient='index')
    df_plot.index.name = 'Group'
    df_plot = df_plot.reset_index().melt(id_vars='Group', var_name='Outcome', value_name='Rate')

    plt.figure(figsize=(10, 6))
    sns.barplot(x='Group', y='Rate', hue='Outcome', data=df_plot, palette='Paired')
    plt.title(title)
    plt.ylabel('Rate (%)')
    plt.ylim(0, 1)
    plt.yticks(np.arange(0, 1.1, 0.1), [f'{int(x*100)}%' for x in np.arange(0, 1.1, 0.1)])
    plt.show()

plot_group_outcomes(gender_disparities, 'Acceptance/Rejection Rates by Gender')
plot_group_outcomes(age_group_disparities, 'Acceptance/Rejection Rates by Age Group')

print("Visualizations generated successfully.")

# Extract metrics specifically for the report to resolve the NameError
gender_metrics = calculate_all_metrics(y_surrogate, y_pred_full_surrogate, blackbox_df['Gender'])
age_sub_df = blackbox_df[blackbox_df['AgeGroup'].isin(['Youth', 'Adult'])]
age_metrics = calculate_all_metrics(y_surrogate[age_sub_df.index], y_pred_full_surrogate[age_sub_df.index], age_sub_df['AgeGroup'])

# Helper for equalized odds which returns a tuple
gender_eodds = equalized_odds(y_surrogate, y_pred_full_surrogate, blackbox_df['Gender'])[0]
age_eodds = equalized_odds(y_surrogate[age_sub_df.index], y_pred_full_surrogate[age_sub_df.index], age_sub_df['AgeGroup'])[0]

# Compile all findings into a structured report
final_report = {
    "Surrogate Model Performance": {
        "Accuracy": f"{accuracy:.4f}",
        "Precision": f"{precision:.4f}",
        "Recall": f"{recall:.4f}",
        "F1-score": f"{f1:.4f}"
    },
    "Fairness Metrics": {
        "Gender": {
            "Demographic Parity Difference": f"{gender_metrics['demographic_parity_difference']:.4f}",
            "Equal Opportunity Difference": f"{gender_metrics['equal_opportunity_difference']:.4f}",
            "Equalized Odds (TPR Diff)": f"{gender_eodds[0]:.4f}",
            "Equalized Odds (FPR Diff)": f"{gender_eodds[1]:.4f}",
            "Disparate Impact Ratio": f"{gender_metrics['disparate_impact']:.4f}"
        },
        "Age (Youth vs Adult)": {
            "Demographic Parity Difference": f"{age_metrics['demographic_parity_difference']:.4f}",
            "Equal Opportunity Difference": f"{age_metrics['equal_opportunity_difference']:.4f}",
            "Equalized Odds (TPR Diff)": f"{age_eodds[0]:.4f}",
            "Equalized Odds (FPR Diff)": f"{age_eodds[1]:.4f}",
            "Disparate Impact Ratio": f"{age_metrics['disparate_impact']:.4f}"
        }
    },
    "Group Disparity Analysis": {
        "Gender Selection Rates": gender_selection_df.set_index('Gender').to_dict('index'),
        "Age Group Selection Rates": age_selection_df.set_index('AgeGroup').to_dict('index'),
        "Worst Affected Group (Gender)": f"{worst_gender['Gender']} with {worst_gender['Selection Rate']:.2f}% selection rate",
        "Worst Affected Group (Age)": f"{worst_age_group['AgeGroup']} with {worst_age_group['Selection Rate']:.2f}% selection rate"
    },
    "SHAP Explainability Analysis": {
        "Top 10 Influential Features": shap_importance.head(10).to_dict('records'),
        "Proxy Discrimination Check": "See printed analysis above for details on sensitive attributes in top SHAP features."
    }
}

import json
print("\n--- FINAL BLACK-BOX FAIRNESS AUDIT REPORT ---")
print(json.dumps(final_report, indent=4))

print("\nBlack-box fairness auditing framework complete. The report provides a comprehensive overview of the surrogate model's performance and fairness assessment.\n")


**Findings from white and black** **box**

In [ ]:
print('\n--- Mean Absolute SHAP Value Comparison ---')

# Check if necessary SHAP values are defined from preceding cells
if 'shap_values_xgb_wb' not in locals() and 'shap_values_xgb_wb' not in globals():
    print("Error: 'shap_values_xgb_wb' not found. Please ensure cell SljAEprdgpk6 is executed completely.")
elif 'shap_values_mlp' not in locals() and 'shap_values_mlp' not in globals():
    print("Error: 'shap_values_mlp' not found. Please ensure cell SljAEprdgpk6 is executed completely.")
elif 'shap_values' not in locals() and 'shap_values' not in globals():
    print("Error: 'shap_values' not found. Please ensure cell QaLnFnLniCvD is executed completely.")
elif 'feature_names_out' not in locals() and 'feature_names_out' not in globals():
    print("Error: 'feature_names_out' not found. Please ensure cell QaLnFnLniCvD is executed completely.")
else:
    # Define mappings for consistent feature names across models
    # This helps in aligning features that might have slightly different names or casing
    # in different parts of the pipeline or datasets.
    feature_name_mapping_wb = {
        'EducationLevel': 'Education_Level',
        'ExperienceYears': 'Experience_Years',
        'PreviousCompanies': 'Previous_Companies',
        'SkillScore': 'Skill_Score',
        'InterviewScore': 'Interview_Score',
        'DistanceFromCompany': 'Distance_From_Company',
        'PersonalityScore': 'Personality_Score',
        # 'Gender' and 'Age' are generally consistent
        # 'RecruitmentStrategy' is specific to the white-box dataset
    }
    feature_name_mapping_mlp = {
        'ExperienceYears': 'Experience_Years',
        'DistanceFromCompany': 'Distance_From_Company',
        'PersonalityScore': 'Personality_Score',
        'SkillScore': 'Skill_Score',
        'InterviewScore': 'Interview_Score',
        # 'Age' is generally consistent
    }
    # Surrogate aggregated features already have good names, but the aggregation function handles this.

    # Filter X_test_wb to exclude the 'Gender_Education_Level' column, which was added post-training
    # This ensures that the features passed to SHAP for xgb_model_wb match the model's training features.
    X_test_wb_filtered_for_shap = X_test_wb.drop(columns=['Gender_Education_Level'], errors='ignore')

    # --- 1. Extract Mean Absolute SHAP values for XGBoost (White-Box) ---
    explainer = shap.TreeExplainer(xgb_model_wb)
    shap_values_xgb_wb = explainer.shap_values(X_test_wb_filtered_for_shap) # Renamed variable

    if isinstance(shap_values_xgb_wb, list): # For multi-output models, take positive class (index 1)
        mean_abs_shap_xgb = np.abs(shap_values_xgb_wb[1]).mean(axis=0)
    else:
        mean_abs_shap_xgb = np.abs(shap_values_xgb_wb).mean(axis=0)

    shap_data_xgb = {}
    for i, feature in enumerate(X_test_wb_filtered_for_shap.columns): # Use filtered columns here too
        canonical_feature = feature_name_mapping_wb.get(feature, feature)
        shap_data_xgb[canonical_feature] = mean_abs_shap_xgb[i]

    # --- 2. Extract Mean Absolute SHAP values for MLP (White-Box) ---
    # MLP was trained only on numerical features after categorical ones were dropped from X_train_scaled
    if isinstance(shap_values_mlp, list): # KernelExplainer often returns a list for multi-output
        mean_abs_shap_mlp = np.abs(shap_values_mlp[0]).mean(axis=0) # Assuming single output for predict
    else:
        mean_abs_shap_mlp = np.abs(shap_values_mlp).mean(axis=0)

    shap_data_mlp = {}
    for i, feature in enumerate(X_test_mlp_processed.columns):
        canonical_feature = feature_name_mapping_mlp.get(feature, feature)
        shap_data_mlp[canonical_feature] = mean_abs_shap_mlp[i]

    # --- 3. Extract Mean Absolute SHAP values for Surrogate (Black-Box) ---
    if isinstance(shap_values, list): # These are for the surrogate model from cell QaLnFnLniCvD
        mean_abs_shap_surrogate = np.abs(shap_values[1]).mean(axis=0) # Take positive class if list
    else:
        mean_abs_shap_surrogate = np.abs(shap_values).mean(axis=0)

    # Create a temporary DataFrame for aggregation
    shap_df_surrogate_direct = pd.DataFrame({
        'Feature': feature_names_out, # feature_names_out comes from surrogate preprocessor
        'Mean_Abs_SHAP': mean_abs_shap_surrogate
    })

    # Helper function to aggregate SHAP values for one-hot encoded features back to original categories
    def aggregate_shap_for_categorical_updated(shap_df, original_categorical_cols, numerical_prefix='num__', categorical_prefix='cat__'):
        aggregated_shap = {}

        for _, row in shap_df.iterrows():
            feature_full_name = row['Feature']
            shap_val = row['Mean_Abs_SHAP']

            # Handle numerical features (prefixed with 'num__')
            if feature_full_name.startswith(numerical_prefix):
                original_name = feature_full_name[len(numerical_prefix):]
                aggregated_shap[original_name] = aggregated_shap.get(original_name, 0) + shap_val
            # Handle categorical features (prefixed with 'cat__')
            elif feature_full_name.startswith(categorical_prefix):
                found_original_cat = False
                for original_col in original_categorical_cols:
                    # Example: cat__Gender_Female, original_col = Gender
                    if feature_full_name.startswith(f'{categorical_prefix}{original_col}_'):
                        aggregated_shap[original_col] = aggregated_shap.get(original_col, 0) + shap_val
                        found_original_cat = True
                        break
                if not found_original_cat:
                    # If not matched to an original_categorical_col, treat as standalone (should not happen with proper setup)
                    aggregated_shap[feature_full_name] = aggregated_shap.get(feature_full_name, 0) + shap_val
            else:
                # Features without specific prefixes (should not happen if preprocessing is consistent)
                aggregated_shap[feature_full_name] = aggregated_shap.get(feature_full_name, 0) + shap_val

        return aggregated_shap # Return dictionary directly

    # `categorical_cols_surrogate` is defined in cell QaLnFnLniCvD and is available globally.
    shap_data_surrogate = aggregate_shap_for_categorical_updated(shap_df_surrogate_direct, categorical_cols_surrogate)

    # --- 4. Combine into a single DataFrame for comparison ---
    # Get all unique canonical feature names from all models' SHAP data
    all_canonical_features = set(shap_data_xgb.keys()) \
                             .union(set(shap_data_mlp.keys())) \
                             .union(set(shap_data_surrogate.keys()))

    # Create the combined DataFrame by iterating through all unique features
    combined_shap_data = []
    for feature in sorted(list(all_canonical_features)):
        row = {
            'Feature': feature,
            'Mean_Abs_SHAP_XGBoost': shap_data_xgb.get(feature, 0.0),
            'Mean_Abs_SHAP_MLP': shap_data_mlp.get(feature, 0.0),
            'Mean_Abs_SHAP_Surrogate': shap_data_surrogate.get(feature, 0.0)
        }
        combined_shap_data.append(row)

    all_models_shap_comparison = pd.DataFrame(combined_shap_data)

    # Sort by overall importance for better visualization
    all_models_shap_comparison['Overall_Mean_Abs_SHAP'] = all_models_shap_comparison[['Mean_Abs_SHAP_XGBoost', 'Mean_Abs_SHAP_MLP', 'Mean_Abs_SHAP_Surrogate']].mean(axis=1)
    all_models_shap_comparison = all_models_shap_comparison.sort_values(by='Overall_Mean_Abs_SHAP', ascending=False).drop(columns=['Overall_Mean_Abs_SHAP']).reset_index(drop=True)

    print("Mean Absolute SHAP values across XGBoost, MLP, and Surrogate Models:")
    display(all_models_shap_comparison.head(15))

    # --- 5. Visualize the comparison ---
    plt.figure(figsize=(14, 8))
    all_models_shap_comparison.set_index('Feature').head(15).plot(kind='bar', figsize=(14, 8))
    plt.title('Top 15 Feature Importance Comparison (Mean Absolute SHAP) Across Models')
    plt.ylabel('Mean Absolute SHAP Value')
    plt.xlabel('Feature')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### Combined Bias Scores (Average DPD) for Sensitive Attributes

### Code for Calculating Inherent Dataset Bias

The `Inherent Dataset Bias` is determined by comparing the ground truth labels (`y_test_wb`) against themselves using the fairness evaluation engine. This highlights any disparities that are already present in the raw data for a given sensitive attribute.

In [ ]:
fairness_engine = FairnessEvaluationEngine()

# Assuming 'suggestions' and 'X_test_wb', 'y_test_wb' are available from previous cells.
# For demonstration, let's pick the first suggestion as an example.
example_attr = suggestions[0]

print(f"Calculating Inherent Dataset Bias for attribute: '{example_attr}'")

# To get inherent bias, we pass y_test_wb as both y_true and y_pred.
dataset_metrics = fairness_engine.calculate_all_metrics(y_test_wb, y_test_wb, X_test_wb[example_attr])

print("Inherent Dataset Bias Metrics:")
for metric, value in dataset_metrics.items():
    if metric != 'group_metrics': # Exclude group_metrics for a concise overview
        print(f"  {metric}: {value:.4f}")


In [ ]:
import pandas as pd
import numpy as np

# Assuming 'all_attributes_comparison_data' and 'suggestions' are available from previous execution

# Define the metric labels in the order they appear in the comparison dataframes
metric_labels = [
    'Demographic Parity Difference',
    'Statistical Parity Difference',
    'Selection Rate Difference',
    'Disparate Impact',
    'Equal Opportunity Difference',
    'FPR Difference',
    'FNR Difference',
    'Average Odds Difference'
]

# Initialize the dictionary to store comparison data for all attributes
all_attributes_comparison_data = {}

# Instantiate the FairnessEvaluationEngine
# This assumes `FairnessEvaluationEngine` class is defined and available from previous cells (e.e.g., SljAEprdgpk6).
fairness_engine = FairnessEvaluationEngine()

print("\n--- Calculating Comprehensive Bias Transformation Data ---")

for attr in suggestions:
    metrics_dict = {
        'Inherent Dataset Bias': {},
        'White-Box XGBoost': {},
        'White-Box MLP': {},
        'Black-Box Surrogate': {}
    }

    # 1. Inherent Dataset Bias (using y_test_wb as both true and predicted to get dataset disparity)
    try:
        # Ensure X_test_wb[attr] is aligned with y_test_wb
        # We can use the fairness_engine's calculate_all_metrics function.
        # To get inherent bias, we pass y_test_wb as both y_true and y_pred.
        dataset_metrics = fairness_engine.calculate_all_metrics(y_test_wb, y_test_wb, X_test_wb[attr])
        metrics_dict['Inherent Dataset Bias'] = {
            'Demographic Parity Difference': dataset_metrics['demographic_parity_difference'],
            'Statistical Parity Difference': dataset_metrics['statistical_parity_difference'],
            'Selection Rate Difference': dataset_metrics['selection_rate_difference'],
            'Disparate Impact': dataset_metrics['disparate_impact'],
            'Equal Opportunity Difference': dataset_metrics['equal_opportunity_difference'],
            'FPR Difference': dataset_metrics['false_positive_rate_difference'],
            'FNR Difference': dataset_metrics['false_negative_rate_difference'],
            'Average Odds Difference': dataset_metrics['average_odds_difference']
        }
    except Exception as e:
        print(f"Warning: Could not calculate Inherent Dataset Bias for '{attr}': {e}")

    # 2. White-Box XGBoost
    try:
        xgb_metrics = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_xgb_wb, X_test_wb[attr])
        metrics_dict['White-Box XGBoost'] = {
            'Demographic Parity Difference': xgb_metrics['demographic_parity_difference'],
            'Statistical Parity Difference': xgb_metrics['statistical_parity_difference'],
            'Selection Rate Difference': xgb_metrics['selection_rate_difference'],
            'Disparate Impact': xgb_metrics['disparate_impact'],
            'Equal Opportunity Difference': xgb_metrics['equal_opportunity_difference'],
            'FPR Difference': xgb_metrics['false_positive_rate_difference'],
            'FNR Difference': xgb_metrics['false_negative_rate_difference'],
            'Average Odds Difference': xgb_metrics['average_odds_difference']
        }
    except Exception as e:
        print(f"Warning: Could not calculate White-Box XGBoost metrics for '{attr}': {e}")

    # 3. White-Box MLP
    try:
        mlp_metrics = fairness_engine.calculate_all_metrics(y_test_wb, y_pred_mlp_wb, X_test_wb[attr])
        metrics_dict['White-Box MLP'] = {
            'Demographic Parity Difference': mlp_metrics['demographic_parity_difference'],
            'Statistical Parity Difference': mlp_metrics['statistical_parity_difference'],
            'Selection Rate Difference': mlp_metrics['selection_rate_difference'],
            'Disparate Impact': mlp_metrics['disparate_impact'],
            'Equal Opportunity Difference': mlp_metrics['equal_opportunity_difference'],
            'FPR Difference': mlp_metrics['false_positive_rate_difference'],
            'FNR Difference': mlp_metrics['false_negative_rate_difference'],
            'Average Odds Difference': mlp_metrics['average_odds_difference']
        }
    except Exception as e:
        print(f"Warning: Could not calculate White-Box MLP metrics for '{attr}': {e}")

    # 4. Black-Box Surrogate (using y_surrogate, y_pred_full_surrogate from QaLnFnLniCvD)
    try:
        # Ensure blackbox_df[attr] is aligned with y_surrogate and y_pred_full_surrogate
        surrogate_metrics = calculate_all_metrics(y_surrogate, y_pred_full_surrogate, blackbox_df[attr]) # Use the calculate_all_metrics from the blackbox section for consistency
        metrics_dict['Black-Box Surrogate'] = {
            'Demographic Parity Difference': surrogate_metrics['demographic_parity_difference'],
            'Statistical Parity Difference': surrogate_metrics['statistical_parity_difference'],
            'Selection Rate Difference': surrogate_metrics['selection_rate_difference'],
            'Disparate Impact': surrogate_metrics['disparate_impact'],
            'Equal Opportunity Difference': surrogate_metrics['equal_opportunity_difference'],
            'FPR Difference': surrogate_metrics['false_positive_rate_difference'],
            'FNR Difference': surrogate_metrics['false_negative_rate_difference'],
            'Average Odds Difference': surrogate_metrics['average_odds_difference']
        }
    except Exception as e:
        print(f"Warning: Could not calculate Black-Box Surrogate metrics for '{attr}': {e}")

    # Create DataFrame for current attribute's comparison
    comparison_df_current_attr = pd.DataFrame(metrics_dict)
    all_attributes_comparison_data[attr] = comparison_df_current_attr
    print(f"Calculated comparison data for attribute: '{attr}'")

print("--- Comprehensive Bias Transformation Data Calculation Complete ---")

# Identify attributes that were already displayed in the previous truncated output
processed_attributes = ['Gender', 'Education_Level'] # Based on the previous standard_output
remaining_attributes = [attr for attr in suggestions if attr not in processed_attributes]

print("\n--- Continuing Comprehensive Bias Transformation Analysis for Remaining Sensitive Attributes ---")

for current_attr in remaining_attributes:
    if current_attr in all_attributes_comparison_data:
        comparison_df_current_attr = all_attributes_comparison_data[current_attr].copy() # Use .copy() to avoid SettingWithCopyWarning

        # Explicitly set the index labels before printing
        comparison_df_current_attr.index = metric_labels

        print(f"\n### Fairness Metric Comparison for '{current_attr}':")
        # Use to_markdown with index=True to ensure full table output with metric names
        print(comparison_df_current_attr.round(4).to_markdown(index=True))

    else:
        print(f"No data found for attribute '{current_attr}' in all_attributes_comparison_data.")

print("### Fairness Metric Comparison for 'Gender':")
# Remove the 'Inherent Dataset Bias' column if it exists
comparison_df_gender = all_attributes_comparison_data['Gender'].copy()
# Explicitly set the index labels before printing
comparison_df_gender.index = metric_labels
print(comparison_df_gender.round(4).to_markdown(index=True))

print("### Fairness Metric Comparison for 'Education_Level':")
# Remove the 'Inherent Dataset Bias' column if it exists
comparison_df_education = all_attributes_comparison_data['Education_Level'].copy()
# Explicitly set the index labels before printing
comparison_df_education.index = metric_labels
print(comparison_df_education.round(4).to_markdown(index=True))
print("\n--- Comprehensive Bias Transformation Analysis Complete ---")

### Combined Bias Score Calculation and Ranking

To identify attributes where bias consistently appears across both white-box (model logic) and black-box (systemic outcomes) environments, we calculate a **Combined Bias Score**. This score aggregates the Demographic Parity Differences (DPD) from each model, providing a holistic view of fairness challenges.

We will consider the average DPD across the XGBoost, MLP, and Surrogate models for each sensitive attribute.

In [ ]:
def calculate_combined_dpd_bias_score(comparison_data_for_attr):
    # Extract Demographic Parity Difference for each model
    dpd_xgb = comparison_data_for_attr.loc['Demographic Parity Difference', 'White-Box XGBoost']
    dpd_mlp = comparison_data_for_attr.loc['Demographic Parity Difference', 'White-Box MLP']
    dpd_surrogate = comparison_data_for_attr.loc['Demographic Parity Difference', 'Black-Box Surrogate']

    # Calculate the average DPD across all three models
    combined_dpd = np.mean([abs(dpd_xgb), abs(dpd_mlp), abs(dpd_surrogate)])
    return combined_dpd

# Initialize a list to store combined bias scores for all attributes
combined_bias_scores = []

for attr in suggestions:
    if attr in all_attributes_comparison_data:
        comparison_df_attr = all_attributes_comparison_data[attr].copy()

        # Ensure the index is correctly set for lookup if it was reset
        if not isinstance(comparison_df_attr.index, pd.Index) or 'Demographic Parity Difference' not in comparison_df_attr.index:
            # Assuming 'metric_labels' from qGOhVf0Yvoqf define the index order
            comparison_df_attr.index = metric_labels

        combined_dpd = calculate_combined_dpd_bias_score(comparison_df_attr)
        combined_bias_scores.append({'Attribute': attr, 'Combined_DPD_Bias_Score': combined_dpd})

# Create a DataFrame from the combined bias scores and rank them
combined_bias_df = pd.DataFrame(combined_bias_scores).sort_values(by='Combined_DPD_Bias_Score', ascending=False).reset_index(drop=True)

print("--- Combined Bias Scores (Average DPD) for Sensitive Attributes ---")
display(combined_bias_df)

# --- Automated Intervention Trigger (Fairness Debt Alerts) ---
combined_bias_threshold = 0.15 # As defined in the problem statement

high_priority_attributes = combined_bias_df[combined_bias_df['Combined_DPD_Bias_Score'] >= combined_bias_threshold]

if not high_priority_attributes.empty:
    print(f"\nALERT: The following attributes exceed the Combined Bias Threshold of {combined_bias_threshold:.2f} and require immediate intervention:")
    display(high_priority_attributes)
else:
    print(f"\nNo attributes exceeded the Combined Bias Threshold of {combined_bias_threshold:.2f}. All identified sensitive attributes are within acceptable bias levels for DPD.")

### Detailed Fairness Metric Comparisons Across Models

This section presents a detailed comparison of various fairness metrics (Demographic Parity Difference, Equal Opportunity Difference, etc.) for each identified sensitive attribute across the White-Box XGBoost, White-Box MLP, and Black-Box Surrogate models.

Observing these metrics helps us understand how bias manifests and potentially changes across different stages of the model development and deployment pipeline.

In [ ]:
print('--- Analysis of Bias Trends Across White-Box and Black-Box Models ---')

for attr in suggestions:
    if attr in all_attributes_comparison_data:
        comparison_df_attr = all_attributes_comparison_data[attr].copy()

        # Set index for lookup
        comparison_df_attr.index = metric_labels

        # Extract Demographic Parity Difference for each stage
        dpd_dataset = abs(comparison_df_attr.loc['Demographic Parity Difference', 'Inherent Dataset Bias'])
        dpd_xgb = abs(comparison_df_attr.loc['Demographic Parity Difference', 'White-Box XGBoost'])
        dpd_mlp = abs(comparison_df_attr.loc['Demographic Parity Difference', 'White-Box MLP'])
        dpd_surrogate = abs(comparison_df_attr.loc['Demographic Parity Difference', 'Black-Box Surrogate'])

        print(f"\nAttribute: {attr}")
        print(f"  1. Inherent Dataset Bias (Baseline): {dpd_dataset:.4f}")

        # Dataset -> XGBoost
        trend_xgb = "Decreased" if dpd_xgb < dpd_dataset else "Increased"
        print(f"  2. White-Box XGBoost: {dpd_xgb:.4f} ({trend_xgb} vs Dataset)")

        # XGBoost -> MLP
        trend_mlp = "Decreased" if dpd_mlp < dpd_xgb else "Increased"
        print(f"  3. White-Box MLP: {dpd_mlp:.4f} ({trend_mlp} vs XGBoost)")

        # MLP -> Black-Box
        trend_bb = "Decreased" if dpd_surrogate < dpd_mlp else "Increased"
        print(f"  4. Black-Box Surrogate: {dpd_surrogate:.4f} ({trend_bb} vs MLP)")

        # Final System Observation
        if dpd_surrogate < dpd_dataset:
            overall_trend = "mitigated compared to the raw dataset"
        else:
            overall_trend = "amplified or remained consistent compared to the raw dataset"
        print(f"  Observation: Overall system bias is {overall_trend}.")
    else:
        print(f"\nWarning: No comparison data found for attribute '{attr}'.")

print('\n--- Overall Summary ---')
print("This analysis shows the step-by-step evolution of Demographic Parity Difference for each sensitive attribute. By comparing each stage, we can pinpoint exactly where bias is being filtered out or reintroduced by the system's logic.")

### **Precise Bias Progression Audit**
This analysis provides an exact tracking of Demographic Parity Difference (DPD) across the recruitment lifecycle, measuring the specific delta at each transition point.

In [ ]:
import pandas as pd
import numpy as np # Ensure numpy is imported for np.argmax

precision_report = []

for attr in suggestions:
    if attr in all_attributes_comparison_data:
        comp = all_attributes_comparison_data[attr]

        # Exact Metrics
        d_dpd = abs(comp.loc['Demographic Parity Difference', 'Inherent Dataset Bias'])
        x_dpd = abs(comp.loc['Demographic Parity Difference', 'White-Box XGBoost'])
        # m_dpd = abs(comp.loc['Demographic Parity Difference', 'White-Box MLP']) # MLP DPD is no longer needed for a direct shift to Black-Box from XGBoost
        s_dpd = abs(comp.loc['Demographic Parity Difference', 'Black-Box Surrogate'])

        # Precise Deltas
        delta_dataset_xgb = x_dpd - d_dpd
        # delta_xgb_mlp = m_dpd - x_dpd # Removed
        delta_xgb_surrogate = s_dpd - x_dpd # New calculation: XGBoost to System
        # delta_mlp_surrogate = s_dpd - m_dpd # Removed
        total_net_change = s_dpd - d_dpd

        precision_report.append({
            'Attribute': attr,
            'Initial (Dataset)': f"{d_dpd:.4f}",
            'XGBoost Shift': f"{delta_dataset_xgb:+.4f}",
            'XGBoost to System Shift (BB)': f"{delta_xgb_surrogate:+.4f}", # Renamed and recalculated
            'Net Impact': "Mitigated" if total_net_change < 0 else "Amplified",
            'Blackbox Surrogate Model DPD': f"{s_dpd:.4f}" # Changed from 'Final DPD'
        })

precision_df = pd.DataFrame(precision_report)
display(precision_df)

print("\n--- Precise Qualitative Insights ---")
for index, row in precision_df.iterrows():
    # Convert string-formatted DPDs back to floats for calculations
    initial_dpd_val = float(row['Initial (Dataset)'])
    final_dpd_val = float(row['Blackbox Surrogate Model DPD']) # Changed from 'Final DPD'

    delta_dataset_xgb_val = float(row['XGBoost Shift'])
    delta_xgb_surrogate_val = float(row['XGBoost to System Shift (BB)'])

    deltas_for_max = [abs(delta_dataset_xgb_val), abs(delta_xgb_surrogate_val)]
    stage_names_for_max = ['Dataset to XGBoost', 'XGBoost to System (BB)']

    max_delta_index = np.argmax(deltas_for_max)
    largest_shift_stage_name = stage_names_for_max[max_delta_index]

    net_impact_status = "Mitigated" if (final_dpd_val - initial_dpd_val) < 0 else "Amplified"
    print(f"* {row['Attribute']}: The system {net_impact_status} bias by {abs(initial_dpd_val - final_dpd_val):.4f}. The largest shift occurred during the {largest_shift_stage_name} stage.")

### Comparing SHAP Values Across Models

This section compares the mean absolute SHAP values for the White-Box XGBoost, White-Box MLP, and Black-Box Surrogate models. This comparison helps in understanding which features are deemed important by each model and reveals consistency or divergence in their decision-making process.

### Detailed SHAP Value Comparison Across Models

This section provides a more in-depth look at the Mean Absolute SHAP values for each feature, allowing us to compare their importance across the White-Box XGBoost, White-Box MLP, and Black-Box Surrogate models. This helps in understanding consistency or divergence in how each model perceives feature importance.

In [ ]:
print('--- Analysis of Feature Importance Trends Across White-Box and Black-Box Models ---')

display(all_models_shap_comparison.head(10))

for index, row in all_models_shap_comparison.iterrows():
    feature = row['Feature']
    shap_xgb = row['Mean_Abs_SHAP_XGBoost']
    shap_mlp = row['Mean_Abs_SHAP_MLP']
    shap_surrogate = row['Mean_Abs_SHAP_Surrogate']

    # Only discuss features that have some importance in at least one model
    if shap_xgb > 0.01 or shap_mlp > 0.01 or shap_surrogate > 0.01:
        print(f"\nFeature: {feature}")
        print(f"  Mean Abs SHAP (White-Box XGBoost): {shap_xgb:.4f}")
        print(f"  Mean Abs SHAP (White-Box MLP): {shap_mlp:.4f}")
        print(f"  Mean Abs SHAP (Black-Box Surrogate): {shap_surrogate:.4f}")

        # Analyze trend from White-Box XGBoost to Black-Box Surrogate
        if shap_xgb > 0 and shap_surrogate > 0:
            if shap_surrogate > shap_xgb * 1.2: # Significant increase
                xgb_trend = "significantly increased"
            elif shap_surrogate < shap_xgb * 0.8: # Significant decrease
                xgb_trend = "significantly decreased"
            elif abs(shap_surrogate - shap_xgb) / shap_xgb < 0.1: # Relatively similar
                xgb_trend = "remained largely consistent"
            else:
                xgb_trend = "changed somewhat"
            print(f"  Importance from White-Box XGBoost to Black-Box Surrogate {xgb_trend}.")

        # Analyze trend from White-Box MLP to Black-Box Surrogate
        if shap_mlp > 0 and shap_surrogate > 0:
            if shap_surrogate > shap_mlp * 1.2:
                mlp_trend = "significantly increased"
            elif shap_surrogate < shap_mlp * 0.8:
                mlp_trend = "significantly decreased"
            elif abs(shap_surrogate - shap_mlp) / shap_mlp < 0.1:
                mlp_trend = "remained largely consistent"
            else:
                mlp_trend = "changed somewhat"
            print(f"  Importance from White-Box MLP to Black-Box Surrogate {mlp_trend}.")
        elif shap_mlp == 0 and shap_surrogate > 0.01: # MLP had no importance, surrogate does
            print(f"  **Observation**: This feature had low importance for MLP but gained notable importance in the Black-Box Surrogate.")

print('\n--- Overall Summary ---')
print("This analysis shows the varying trends in Mean Absolute SHAP values for each feature when comparing the white-box models (XGBoost and MLP) with the black-box surrogate model. It highlights instances where a feature's perceived importance is amplified or mitigated as we move from the explicit model logic to the observed system behavior. Discrepancies can indicate that the black-box system might be relying on different features or combinations of features than initially expected from the white-box models.")

### Evolution of Demographic Parity Difference Across Models

This visualization illustrates how the Demographic Parity Difference (DPD) for each sensitive attribute changes from the initial dataset, through the White-Box XGBoost and MLP models, and finally to the Black-Box Surrogate model. A higher absolute DPD value indicates a greater disparity in selection rates between groups for that attribute. This helps in understanding at which stage bias is introduced, amplified, or mitigated.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print('\n--- Visualizing Demographic Parity Difference Evolution ---')

dpd_evolution_data = []

# 'suggestions' variable from previous cells contains the list of sensitive attributes
for attr in suggestions:
    if attr in all_attributes_comparison_data:
        attr_metrics = all_attributes_comparison_data[attr]
        # Extract DPD values (absolute for magnitude comparison)
        dpd_dataset = abs(attr_metrics.loc['Demographic Parity Difference', 'Inherent Dataset Bias'])
        dpd_xgb = abs(attr_metrics.loc['Demographic Parity Difference', 'White-Box XGBoost'])
        dpd_mlp = abs(attr_metrics.loc['Demographic Parity Difference', 'White-Box MLP'])
        dpd_surrogate = abs(attr_metrics.loc['Demographic Parity Difference', 'Black-Box Surrogate'])

        dpd_evolution_data.append({
            'Attribute': attr,
            'Stage': 'Dataset',
            'DPD': dpd_dataset
        })
        dpd_evolution_data.append({
            'Attribute': attr,
            'Stage': 'XGBoost (White-Box)',
            'DPD': dpd_xgb
        })
        dpd_evolution_data.append({
            'Attribute': attr,
            'Stage': 'MLP (White-Box)',
            'DPD': dpd_mlp
        })
        dpd_evolution_data.append({
            'Attribute': attr,
            'Stage': 'Surrogate (Black-Box)',
            'DPD': dpd_surrogate
        })

dpd_evolution_df = pd.DataFrame(dpd_evolution_data)

# Define the order of stages for plotting
stage_order = ['Dataset', 'XGBoost (White-Box)', 'MLP (White-Box)', 'Surrogate (Black-Box)']
dpd_evolution_df['Stage'] = pd.Categorical(dpd_evolution_df['Stage'], categories=stage_order, ordered=True)

plt.figure(figsize=(8, 6))
sns.lineplot(data=dpd_evolution_df, x='Stage', y='DPD', hue='Attribute', marker='o', palette='tab10')
plt.title('Evolution of Absolute Demographic Parity Difference Across System Stages')
plt.xlabel('System Stage')
plt.ylabel('Absolute Demographic Parity Difference (DPD)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\n--- DPD Evolution Visualization Complete ---')

# Task
**1. Explanation of DPD Evolution Visualization**

The 'Evolution of Absolute Demographic Parity Difference Across System Stages' visualization provides critical insights into how bias, specifically Demographic Parity Difference (DPD), is introduced, amplified, or mitigated throughout the recruitment system. Here are the key observations for each sensitive attribute:

*   **Gender**:
    *   **Initial Bias (Dataset)**: Gender starts with a high absolute DPD (approx. 0.116), indicating a significant initial disparity in selection rates within the raw dataset.
    *   **White-Box Models (XGBoost, MLP)**: The DPD for Gender slightly decreases in both the XGBoost (approx. 0.113) and MLP (approx. 0.096) models. This suggests that the internal logic of these models might be marginally reducing the DPD compared to the raw data.
    *   **Black-Box System (Surrogate)**: A notable reduction in DPD (approx. 0.062) is observed in the Black-Box Surrogate model. This indicates that the overall observed behavior of the system, including any expert rules or post-processing, significantly mitigates the DPD related to Gender compared to the initial dataset and the individual white-box models.

*   **Education_Level**:
    *   **Initial Bias (Dataset)**: Education_Level also exhibits a substantial initial DPD (approx. 0.103).
    *   **White-Box Models (XGBoost, MLP)**: Similar to Gender, the DPD for Education_Level shows a slight decrease in the white-box models (both approx. 0.099).
    *   **Black-Box System (Surrogate)**: Further mitigation is observed in the Black-Box Surrogate model, with the DPD decreasing to approximately 0.077. This again suggests an overall mitigating effect within the system's observed outcomes.

*   **Previous_Companies**:
    *   **Initial Bias (Dataset)**: This attribute starts with a moderate DPD (approx. 0.039).
    *   **White-Box Models (XGBoost, MLP)**: The DPD remains largely consistent in the white-box models (both approx. 0.039).
    *   **Black-Box System (Surrogate)**: A decrease in DPD to approximately 0.023 is seen in the Black-Box Surrogate.

*   **Location**:
    *   **Initial Bias (Dataset)**: Location has a relatively lower initial DPD (approx. 0.025).
    *   **White-Box Models (XGBoost, MLP)**: The white-box models significantly reduce the DPD (both approx. 0.013) compared to the raw dataset. This indicates that the model's logic itself is effective in mitigating this particular bias.
    *   **Black-Box System (Surrogate)**: The DPD slightly decreases further to approximately 0.011 in the Black-Box Surrogate.

*   **Job_Role_Applied**:
    *   **Initial Bias (Dataset)**: This attribute shows the lowest initial DPD (approx. 0.006).
    *   **White-Box Models (XGBoost, MLP)**: The DPD remains very low in the white-box models (XGBoost approx. 0.008, MLP approx. 0.006).
    *   **Black-Box System (Surrogate)**: Notably, the DPD for Job_Role_Applied *increases* to approximately 0.011 in the Black-Box Surrogate model. This is an instance of **bias amplification** within the overall system's observed behavior, even though the initial bias was low. This indicates that the interplay of models and expert rules in the black-box system might be introducing or amplifying DPD for job roles.

**Summary of DPD Evolution:**
For most attributes (Gender, Education_Level, Previous_Companies, Location), the system exhibits a trend of **bias mitigation**, where the DPD gradually decreases from the raw dataset through the white-box models to the final black-box surrogate. However, for 'Job_Role_Applied', the DPD is **amplified** in the black-box stage, suggesting a need for closer examination of this particular attribute's treatment within the full system.

---

**2. Additional Bias Detection Ideas**

Building on the current comprehensive analysis, here are further ideas to detect and analyze bias:

*   **Intersectionality Analysis**:
    *   **Method**: Instead of analyzing each sensitive attribute in isolation, compute fairness metrics (like DPD, Equal Opportunity Difference, FNR/FPR differences) for combinations of attributes. For example, analyze "Female applicants with a High School education level" vs. "Male applicants with a Masters degree".
    *   **Insight**: This approach uncovers compounded biases where individuals belonging to multiple marginalized groups face disproportionately worse outcomes than those marginalized by only one attribute or none. This is crucial for understanding nuanced forms of discrimination.
    *   **Visualization**: Stacked bar charts or heatmaps showing DPD (or other metrics) across different intersectional groups.

*   **Error Disparity Visualization**:
    *   **Method**: While we currently plot TPR, FPR, and FNR by group, a dedicated visualization that explicitly compares error types across groups for a chosen sensitive attribute can be highly informative. For instance, create a side-by-side comparison of confusion matrices or stacked bar charts showing the proportion of True Positives, False Positives, True Negatives, and False Negatives for each group within a sensitive attribute.
    *   **Insight**: This allows for a deeper understanding of *which types* of errors (e.g., being unfairly rejected - False Negative, or being wrongly selected - False Positive) disproportionately affect different groups. For example, one group might suffer from a higher FNR (disadvantage in selection), while another suffers from a higher FPR (disadvantage in fairness).
    *   **Visualization**: Grouped bar plots for FPR and FNR across groups, or heatmaps of confusion matrices for each group.

---

**3. Final Task: Bias Detection Phase Summary**

The bias detection phase has revealed critical insights into the fairness of the recruitment system:

1.  **Highest DPD Attributes**: Based on the combined DPD score across all models, **'Gender' (0.0903)** and **'Education_Level' (0.0694)** consistently exhibit the highest levels of Demographic Parity Difference. These attributes require the most immediate attention for potential bias mitigation strategies.
2.  **General Mitigation Trend**: For most sensitive attributes (Gender, Education_Level, Previous_Companies, Location), there's an observable trend of **DPD mitigation** as we move from the raw dataset through the white-box models to the black-box surrogate model. This suggests that the combined system, or at least its observed output, tends to reduce some of the inherent biases present in the initial data.
3.  **Bias Amplification for 'Job_Role_Applied'**: A significant finding is the **amplification of DPD for 'Job_Role_Applied'** in the Black-Box Surrogate model. Although starting with the lowest initial bias, its DPD increases in the final system stage. This indicates that the black-box system's decision-making process, which includes the combination of white-box models and potentially expert rules, introduces or exacerbates disparities related to job roles. This specific amplification warrants a detailed investigation into the black-box components and their interaction with this attribute.
4.  **No High Priority Alerts**: While biases exist, none of the attributes exceeded the defined **Combined Bias Threshold of 0.15**, meaning no immediate "Fairness Debt Alerts" were triggered for DPD, which is a positive sign. However, the identified DPDs, especially for Gender and Education_Level, still indicate areas for improvement.
5.  **SHAP Insights**: SHAP analysis has provided insights into feature importance, highlighting that `Interview_Score` and `Skill_Score` are consistently highly influential across models, while some features like `Technical_Test_Score` and `Aptitude_Test_Score` gained notable importance in the Black-Box Surrogate model, despite low importance in the white-box models. This discrepancy suggests that the black-box system might be implicitly relying on different or re-weighted feature combinations, potentially influencing bias trends.

In conclusion, the bias detection phase has successfully identified key sensitive attributes with varying levels of disparity, pinpointed stages of bias mitigation and amplification, and provided initial leads for where further investigation and targeted interventions might be most effective.

### Section 3.1: Black-Box Behavioral Consistency & Fidelity
Implementing the **Prediction Consistency Score** and the **Fidelity Metric** ($Matching / Total$) as required by the Black-Box auditing section.

In [ ]:
# 1. Surrogate Model Fidelity Analysis
y_blackbox_actual = blackbox_df['Final Decision_Numeric'].values
y_surrogate_preds = y_pred_full_surrogate # From cell QaLnFnLniCvD

fidelity_score = np.sum(y_blackbox_actual == y_surrogate_preds) / len(y_blackbox_actual)
print(f"Surrogate Model Fidelity Score: {fidelity_score:.42%}")

# 2. Behavioral Consistency Testing
# Generate slightly modified profiles to check for 'Instability'
def check_behavioral_consistency(n_samples=100):
    inconsistent_count = 0
    for _ in range(n_samples):
        base_candidate = generate_candidate()
        res1 = predict_candidate_v2(base_candidate)['Final Decision']

        # Slightly modify a non-sensitive attribute (e.g., Expected Salary +/- 1%)
        mod_candidate = base_candidate.copy()
        mod_candidate['Expected_Salary'] *= 1.01
        res2 = predict_candidate_v2(mod_candidate)['Final Decision']

        if res1 != res2:
            inconsistent_count += 1

    consistency_score = (n_samples - inconsistent_count) / n_samples
    return consistency_score, inconsistent_count

c_score, i_count = check_behavioral_consistency()
print(f"\nBehavioral Consistency Score: {c_score:.2%}")
print(f"Inconsistent Outputs detected: {i_count} out of 100 tests")

### Section 4.1: Counterfactual Discrimination Rate
Quantifying the **Counterfactual Discrimination Rate** across the test set for the primary sensitive attribute.

In [ ]:
def calculate_discrimination_rate(model, X_test, attr_name, n_audit=500):
    violations = 0
    X_audit = X_test.head(n_audit).copy()

    # Ensure categorical types are consistent for prediction
    for col in X_audit.select_dtypes(['object', 'category']).columns:
        X_audit[col] = X_audit[col].astype('category')

    original_preds = model.predict(X_audit)
    unique_vals = X_test[attr_name].unique()

    for i in range(len(X_audit)):
        row = X_audit.iloc[[i]]
        base_pred = original_preds[i]
        current_val = row[attr_name].values[0]

        for val in unique_vals:
            if val == current_val: continue
            cf_row = row.copy()
            cf_row[attr_name] = val
            # Enforce category type after value change
            cf_row[attr_name] = cf_row[attr_name].astype('category')

            if model.predict(cf_row)[0] != base_pred:
                violations += 1
                break

    return violations / n_audit

d_rate = calculate_discrimination_rate(xgb_model_wb, X_test_wb_filtered_for_shap, EXTERNAL_SENSITIVE_ATTR)
print(f"Counterfactual Discrimination Rate for {EXTERNAL_SENSITIVE_ATTR}: {d_rate:.2%}")

### Section 3.2: Sensitivity Analysis Ranking
This section ranks features by their 'Sensitivity' — how much a small perturbation in the feature value changes the model's output probability.

In [ ]:
def calculate_sensitivity_ranking(model, X_test, features):
    sensitivity_scores = {}
    # Ensure X_test is in categorical format for XGBoost
    X_test_clean = X_test.copy()
    for col in X_test_clean.select_dtypes(['object', 'category']).columns:
        X_test_clean[col] = X_test_clean[col].astype('category')

    base_probs = model.predict_proba(X_test_clean)[:, 1]

    for col in features:
        X_perturbed = X_test_clean.copy()
        if X_perturbed[col].dtype.name in ['category', 'object']:
            # For categorical, we pick a random different category from the existing codes
            unique_vals = X_test_clean[col].unique()
            if len(unique_vals) > 1:
                X_perturbed[col] = X_test_clean[col].apply(lambda x: np.random.choice([v for v in unique_vals if v != x]))
                X_perturbed[col] = X_perturbed[col].astype('category')
        else:
            # For numerical, add 5% noise
            std = X_test_clean[col].std()
            X_perturbed[col] = X_test_clean[col] + (0.05 * std)

        new_probs = model.predict_proba(X_perturbed)[:, 1]
        sensitivity_scores[col] = np.mean(np.abs(new_probs - base_probs))

    return pd.DataFrame(list(sensitivity_scores.items()), columns=['Feature', 'Sensitivity_Impact']).sort_values(by='Sensitivity_Impact', ascending=False)

sensitivity_df = calculate_sensitivity_ranking(xgb_model_wb, X_test_wb_filtered_for_shap.head(1000), X_test_wb_filtered_for_shap.columns)
print("--- Sensitivity Analysis Ranking ---")
display(sensitivity_df.head(10))

### 9. Black-Box Fairness Audit (Surrogate Model)
In this section, we evaluate the fairness of the overall system behavior using the Surrogate Model. We focus on 'Equalized Odds' and 'Disparate Impact' to see if the systemic outcomes (including any hidden expert rules) are fair across groups.

In [ ]:
print(f"--- Black-Box Fairness Audit for Attribute: {EXTERNAL_SENSITIVE_ATTR} ---")

# 1. Calculate Black-Box Metrics using the surrogate model predictions
# y_surrogate and y_pred_full_surrogate were defined in the black-box setup (cell QaLnFnLniCvD)
bb_attr_series = blackbox_df[EXTERNAL_SENSITIVE_ATTR]
bb_metrics = calculate_all_metrics(y_surrogate, y_pred_full_surrogate, bb_attr_series)

# 2. Extract specific metrics for interpretation
di_ratio = bb_metrics['disparate_impact']
dp_diff = bb_metrics['demographic_parity_difference']
eod_diff = bb_metrics['equal_opportunity_difference']

print(f"\nMetrics for {EXTERNAL_SENSITIVE_ATTR}:")
print(f"  Disparate Impact Ratio: {di_ratio:.4f} (Ideal: 1.0)")
print(f"  Statistical Parity Difference: {dp_diff:.4f} (Ideal: 0.0)")
print(f"  Equal Opportunity Difference: {eod_diff:.4f} (Ideal: 0.0)")

# 3. Visualization of Disparity
plt.figure(figsize=(10, 6))
sns.barplot(x=gender_selection_df['Gender'], y=gender_selection_df['Selection Rate'], palette='coolwarm')
plt.title(f'Black-Box Selection Rate by {EXTERNAL_SENSITIVE_ATTR}')
plt.ylabel('Selection Rate (%)')
plt.axhline(y=80, color='red', linestyle='--', label='80% Rule (Impact Threshold)')
plt.legend()
plt.show()

### 10. Counterfactual Fairness Testing
Counterfactual fairness asks: 'Would the model's decision change for this individual if their protected attribute (e.g., Gender) were different, while keeping all other features constant?'

In [ ]:
print("--- Counterfactual Fairness Test ---")

# Pick a sample candidate from the test set
sample_idx = 0
# Filter out columns that were not present during XGBoost training (like the intersectional attribute)
train_features = xgb_model_wb.get_booster().feature_names
candidate_cf = X_test_wb.iloc[[sample_idx]][train_features].copy()

# Ensure all object columns are converted to categories as required by the model
for col in candidate_cf.select_dtypes(['object']).columns:
    candidate_cf[col] = candidate_cf[col].astype('category')

original_gender = candidate_cf[EXTERNAL_SENSITIVE_ATTR].values[0]

# Get original prediction
orig_pred = xgb_model_wb.predict(candidate_cf)[0]
orig_label = "Selected" if orig_pred == 1 else "Rejected"

print(f"Original Profile (Gender {original_gender}): {orig_label}")

# Flip the gender and re-predict
counterfactual_results = []
# Iterate through common gender encodings present in the dataset
for g in [0, 1, 2]:
    cf_candidate = candidate_cf.copy()
    cf_candidate[EXTERNAL_SENSITIVE_ATTR] = g
    # Ensure the flipped column is also treated as categorical if the original was
    if candidate_cf[EXTERNAL_SENSITIVE_ATTR].dtype.name == 'category':
        cf_candidate[EXTERNAL_SENSITIVE_ATTR] = cf_candidate[EXTERNAL_SENSITIVE_ATTR].astype('category')

    cf_pred = xgb_model_wb.predict(cf_candidate)[0]
    cf_label = "Selected" if cf_pred == 1 else "Rejected"
    print(f"Counterfactual Profile (Gender {g}): {cf_label}")
    counterfactual_results.append(cf_pred)

if len(set(counterfactual_results)) > 1:
    print("\nRESULT: Model FAILs counterfactual fairness test for this individual.")
else:
    print("\nRESULT: Model PASSes counterfactual fairness test for this individual.")

### 11. SHAP Deep-Dive: Investigating Hidden Proxies

In this section, we use SHAP (SHapley Additive exPlanations) values to determine which features most influence the model's decision and whether features like `Education_Level` or `Experience_Years` are acting as proxies for `Gender`.

In [ ]:
import shap
import matplotlib.pyplot as plt

# Using the surrogate model pipeline from cell QaLnFnLniCvD
xgb_surrogate = surrogate_model_pipeline.named_steps['classifier']
preprocessor = surrogate_model_pipeline.named_steps['preprocessor']

# Transform the surrogate test set
X_test_transformed = preprocessor.transform(X_test_surrogate)
feature_names = preprocessor.get_feature_names_out()

# Initialize TreeExplainer
explainer = shap.TreeExplainer(xgb_surrogate)
shap_values = explainer.shap_values(X_test_transformed)

# Summary Plot to identify influential features
print("SHAP Summary Plot: Feature Importance and Directionality")
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, show=False)
plt.show()

### 12. Contextualizing Disparate Impact

We will now compare the model's Disparate Impact (DI) Ratio against the historical bias in the raw data to see if the model amplifies or mitigates existing disparities.

In [ ]:
# Calculate Historical Disparate Impact
gender_counts = df_inspect.groupby('Gender')['Hiring_Decision'].mean()
privileged_rate = gender_counts.max()
unprivileged_rate = gender_counts.min()
historical_di = unprivileged_rate / privileged_rate

model_di = bb_metrics['disparate_impact']

print(f"Historical Disparate Impact (Raw Data): {historical_di:.4f}")
print(f"Model Disparate Impact (Surrogate): {model_di:.4f}")

amplification = (1 - model_di) / (1 - historical_di) if historical_di != 1 else 0
print(f"Bias Amplification Factor: {amplification:.4f}")

if model_di > historical_di:
    print("Conclusion: The model is mitigating historical bias.")
else:
    print("Conclusion: The model is amplifying historical bias.")

### Top 10 Most Influential Features (Surrogate Model SHAP) Plot

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# The shap_importance DataFrame is available from previous execution
# Display the plot for the top 10 most influential features
plt.figure(figsize=(6, 7.5))
sns.barplot(x='SHAP_Importance', y='Feature', data=shap_importance.head(10), palette='viridis', hue='Feature', legend=False)
plt.title('Top 10 Most Influential Features (Surrogate Model SHAP)')
plt.xlabel('Mean Absolute SHAP Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

#Weight Calculation

In [ ]:
import numpy as np
import pandas as pd

# =====================================================
# AHP Weight Calculation for Fairness Risk Index (FRI)
# =====================================================

criteria = [
    "Discrimination Risk (DR)",
    "Bias Amplification Risk (BAR)",
    "Harm Influence Risk (HIR)",
    "Stability Risk (SR)",
    "Fidelity Risk (FR)"
]

# Pairwise comparison matrix
# Saaty Scale:
# 1 = Equal Importance
# 3 = Moderate Importance
# 5 = Strong Importance
# 7 = Very Strong Importance
# 9 = Extreme Importance

pairwise_matrix = np.array([
    [1,   3,   4,   5,   6],      # DR
    [1/3, 1,   2,   3,   4],      # BAR
    [1/4, 1/2, 1,   2,   3],      # HIR
    [1/5, 1/3, 1/2, 1,   2],      # SR
    [1/6, 1/4, 1/3, 1/2, 1]       # FR
])

# -----------------------------------------------------
# Step 1: Calculate Eigenvalues and Eigenvectors
# -----------------------------------------------------

eigenvalues, eigenvectors = np.linalg.eig(pairwise_matrix)

max_index = np.argmax(eigenvalues.real)
max_eigenvalue = eigenvalues[max_index].real

weights = eigenvectors[:, max_index].real
weights = weights / np.sum(weights)

# -----------------------------------------------------
# Step 2: Consistency Check
# -----------------------------------------------------

n = pairwise_matrix.shape[0]

CI = (max_eigenvalue - n) / (n - 1)

RI_values = {
    1: 0.00,
    2: 0.00,
    3: 0.58,
    4: 0.90,
    5: 1.12,
    6: 1.24,
    7: 1.32,
    8: 1.41,
    9: 1.45,
    10: 1.49
}

RI = RI_values[n]
CR = CI / RI

# -----------------------------------------------------
# Step 3: Display Results
# -----------------------------------------------------

weights_df = pd.DataFrame({
    "Criterion": criteria,
    "Weight": np.round(weights, 4),
    "Percentage": np.round(weights * 100, 2)
})

print("="*60)
print("AHP-DERIVED FAIRNESS RISK INDEX WEIGHTS")
print("="*60)

print(weights_df)

print("\n")
print("="*60)
print("CONSISTENCY ANALYSIS")
print("="*60)

print(f"Maximum Eigenvalue (λmax): {max_eigenvalue:.4f}")
print(f"Consistency Index (CI): {CI:.4f}")
print(f"Consistency Ratio (CR): {CR:.4f}")

if CR < 0.10:
    print("\nConsistency Status: ACCEPTABLE")
    print("The weights are scientifically consistent.")
else:
    print("\nConsistency Status: NOT ACCEPTABLE")
    print("Revise the pairwise comparison matrix.")

print("="*60)

# -----------------------------------------------------
# Step 4: Ready-to-use FRI Formula
# -----------------------------------------------------

print("\nSuggested FRI Formula:")

for c, w in zip(criteria, weights):
    print(f"{w:.4f} × {c}")

print("\nFRI = Sum(Weight × Normalized Risk Component) × 100")

In [ ]:
# ============================================================
# EXPLAINABILITY GUIDED HARM ASSESSMENT (EGHA)
# HS = sqrt(CBS × SIS)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("="*70)
print("EXPLAINABILITY GUIDED HARM ASSESSMENT (EGHA)")
print("="*70)

# ============================================================
# STEP 1 : CREATE SHAP DATAFRAME
# ============================================================

mean_abs_shap = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'Attribute': feature_names,
    'Mean_SHAP': mean_abs_shap
})

# ============================================================
# STEP 2 : AGGREGATE ONE-HOT FEATURES
# ============================================================

aggregated_sis = {}

aggregated_sis['Gender'] = shap_df[
    shap_df['Attribute'].str.contains('Gender', case=False)
]['Mean_SHAP'].sum()

aggregated_sis['Education_Level'] = shap_df[
    shap_df['Attribute'].str.contains('Education_Level', case=False)
]['Mean_SHAP'].sum()

aggregated_sis['Location'] = shap_df[
    shap_df['Attribute'].str.contains('Location', case=False)
]['Mean_SHAP'].sum()

aggregated_sis['Job_Role_Applied'] = shap_df[
    shap_df['Attribute'].str.contains('Job_Role_Applied', case=False)
]['Mean_SHAP'].sum()

aggregated_sis['Previous_Companies'] = shap_df[
    shap_df['Attribute'].str.contains('Previous_Companies', case=False)
]['Mean_SHAP'].sum()

aggregated_sis_df = pd.DataFrame(
    aggregated_sis.items(),
    columns=['Attribute','Mean_SHAP']
)

# ============================================================
# STEP 3 : NORMALIZED SHAP INFLUENCE SCORE
# ============================================================

aggregated_sis_df['SIS'] = (
    aggregated_sis_df['Mean_SHAP']
    /
    aggregated_sis_df['Mean_SHAP'].max()
)

print("\nAGGREGATED SHAP INFLUENCE SCORES")
display(aggregated_sis_df)

# ============================================================
# STEP 4 : ATTRIBUTES TO AUDIT
# ============================================================

audit_attributes = [
    'Gender',
    'Education_Level',
    'Previous_Companies',
    'Location',
    'Job_Role_Applied'
]

# ============================================================
# STEP 5 : CALCULATE CBS + HS
# ============================================================

harm_results = []

for attr in audit_attributes:

    metrics = calculate_all_metrics(
        y_test_wb,
        y_pred_xgb_wb,
        X_test_wb[attr]
    )

    CBS = (
        abs(metrics['demographic_parity_difference']) * 0.20 +
        abs(1 - metrics['disparate_impact']) * 0.20 +
        abs(metrics['equal_opportunity_difference']) * 0.20 +
        abs(metrics['selection_rate_difference']) * 0.10 +
        abs(metrics['false_positive_rate_difference']) * 0.10 +
        abs(metrics['false_negative_rate_difference']) * 0.10 +
        abs(metrics['average_odds_difference']) * 0.05 +
        abs(metrics['statistical_parity_difference']) * 0.05
    )

    SIS = aggregated_sis_df.loc[
        aggregated_sis_df['Attribute'] == attr,
        'SIS'
    ].values[0]

    # ========================================================
    # GEOMETRIC HARM SCORE
    # ========================================================

    HS = np.sqrt(CBS * SIS)

    harm_results.append([
        attr,
        round(CBS,4),
        round(float(SIS),4),
        round(float(HS),4)
    ])

# ============================================================
# STEP 6 : CREATE RESULTS TABLE
# ============================================================

harm_df = pd.DataFrame(
    harm_results,
    columns=[
        'Attribute',
        'CBS',
        'SIS',
        'HS'
    ]
)

harm_df = harm_df.sort_values(
    by='HS',
    ascending=False
)

harm_df.insert(
    0,
    'Rank',
    range(1, len(harm_df)+1)
)

print("\nATTRIBUTE HARM RANKING")
display(harm_df)

# ============================================================
# STEP 7 : HARM LEVEL
# ============================================================

def classify_harm(x):

    if x < 0.20:
        return "Low"

    elif x < 0.40:
        return "Moderate"

    elif x < 0.60:
        return "High"

    else:
        return "Critical"

harm_df['Harm_Level'] = harm_df['HS'].apply(
    classify_harm
)

print("\nFINAL EGHA RESULTS")
display(harm_df)

# ============================================================
# STEP 8 : VISUALIZATION
# ============================================================

plt.figure(figsize=(10,6))

bars = plt.bar(
    harm_df['Attribute'],
    harm_df['HS']
)

for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{height:.3f}',
        ha='center',
        va='bottom'
    )

plt.title(
    'Explainability Guided Harm Assessment (EGHA)'
)

plt.xlabel('Sensitive Attribute')
plt.ylabel('Harm Score (HS)')

plt.xticks(rotation=30)

plt.tight_layout()

plt.show()

# ============================================================
# STEP 9 : MOST HARMFUL ATTRIBUTE
# ============================================================

print("\nMOST HARMFUL ATTRIBUTE")

print(
    f"{harm_df.iloc[0]['Attribute']}"
)

print(
    f"Harm Score = {harm_df.iloc[0]['HS']}"
)

print(
    f"Harm Level = {harm_df.iloc[0]['Harm_Level']}"
)

### Deriving AHP Weights for CBS Metrics

To replace the fixed weights in the Combined Bias Score (CBS) calculation with analytically derived weights, we will use the Analytic Hierarchy Process (AHP). This section guides you through defining the pairwise comparisons for each of the 8 individual fairness metrics that contribute to the CBS. Your judgments will determine the relative importance of each metric.

**Fairness Metrics for CBS:**

1.  **Demographic Parity Difference (DPD)**: Measures the absolute difference in selection rates.
2.  **Disparate Impact (DI)**: Ratio of selection rates (transformed to `abs(1 - DI)` for consistency).
3.  **Equal Opportunity Difference (EOD)**: Difference in True Positive Rates.
4.  **Selection Rate Difference (SRD)**: Similar to DPD, but explicitly named.
5.  **False Positive Rate Difference (FPRD)**: Difference in False Positive Rates.
6.  **False Negative Rate Difference (FNRD)**: Difference in False Negative Rates.
7.  **Average Odds Difference (AOD)**: Average of TPR and FPR differences.
8.  **Statistical Parity Difference (SPD)**: Similar to DPD, often representing the unprivileged minus privileged selection rate.

In [ ]:
import numpy as np
import pandas as pd

# Define the 8 fairness metrics that constitute the CBS
cbs_metrics = [
    "Demographic Parity Difference",
    "Disparate Impact",
    "Equal Opportunity Difference",
    "Selection Rate Difference",
    "False Positive Rate Difference",
    "False Negative Rate Difference",
    "Average Odds Difference",
    "Statistical Parity Difference"
]

n = len(cbs_metrics)

# --- USER INPUT REQUIRED: Fill this pairwise comparison matrix ---
# Use Saaty Scale for comparisons:
# 1 = Equal Importance
# 3 = Moderate Importance
# 5 = Strong Importance
# 7 = Very Strong Importance
# 9 = Extreme Importance
# Reciprocal values (e.g., 1/3, 1/5) for inverse comparisons.

# Initialize with 1s on the diagonal
pairwise_matrix_cbs = np.ones((n, n))

# Example: If DPD is 'moderately more important' than DI, set matrix[0, 1] = 3.0
# Then, matrix[1, 0] automatically becomes 1/3.0.
# You need to fill in the upper triangle (above the diagonal) for each pair.

# Metric 1: Demographic Parity Difference (DPD)
# Example values - **YOU SHOULD REPLACE THESE BASED ON YOUR JUDGMENT**
pairwise_matrix_cbs[0, 1] = 1.0 # DPD vs Disparate Impact
pairwise_matrix_cbs[0, 2] = 2.0 # DPD vs Equal Opportunity Difference
pairwise_matrix_cbs[0, 3] = 1.0 # DPD vs Selection Rate Difference (likely same importance as DPD itself)
pairwise_matrix_cbs[0, 4] = 3.0 # DPD vs False Positive Rate Difference
pairwise_matrix_cbs[0, 5] = 3.0 # DPD vs False Negative Rate Difference
pairwise_matrix_cbs[0, 6] = 2.0 # DPD vs Average Odds Difference
pairwise_matrix_cbs[0, 7] = 1.0 # DPD vs Statistical Parity Difference (often similar to DPD)

# Metric 2: Disparate Impact (DI)
# (Comparisons with DPD already set by reciprocity)
pairwise_matrix_cbs[1, 2] = 1.0 # DI vs Equal Opportunity Difference
pairwise_matrix_cbs[1, 3] = 1.0 # DI vs Selection Rate Difference
pairwise_matrix_cbs[1, 4] = 2.0 # DI vs False Positive Rate Difference
pairwise_matrix_cbs[1, 5] = 2.0 # DI vs False Negative Rate Difference
pairwise_matrix_cbs[1, 6] = 1.0 # DI vs Average Odds Difference
pairwise_matrix_cbs[1, 7] = 1.0 # DI vs Statistical Parity Difference

# Metric 3: Equal Opportunity Difference (EOD)
pairwise_matrix_cbs[2, 3] = 2.0 # EOD vs Selection Rate Difference
pairwise_matrix_cbs[2, 4] = 1.0 # EOD vs False Positive Rate Difference
pairwise_matrix_cbs[2, 5] = 1.0 # EOD vs False Negative Rate Difference
pairwise_matrix_cbs[2, 6] = 1.0 # EOD vs Average Odds Difference
pairwise_matrix_cbs[2, 7] = 2.0 # EOD vs Statistical Parity Difference

# Metric 4: Selection Rate Difference (SRD)
pairwise_matrix_cbs[3, 4] = 3.0 # SRD vs False Positive Rate Difference
pairwise_matrix_cbs[3, 5] = 3.0 # SRD vs False Negative Rate Difference
pairwise_matrix_cbs[3, 6] = 2.0 # SRD vs Average Odds Difference
pairwise_matrix_cbs[3, 7] = 1.0 # SRD vs Statistical Parity Difference

# Metric 5: False Positive Rate Difference (FPRD)
pairwise_matrix_cbs[4, 5] = 1.0 # FPRD vs False Negative Rate Difference
pairwise_matrix_cbs[4, 6] = 1.0 # FPRD vs Average Odds Difference
pairwise_matrix_cbs[4, 7] = 2.0 # FPRD vs Statistical Parity Difference

# Metric 6: False Negative Rate Difference (FNRD)
pairwise_matrix_cbs[5, 6] = 1.0 # FNRD vs Average Odds Difference
pairwise_matrix_cbs[5, 7] = 2.0 # FNRD vs Statistical Parity Difference

# Metric 7: Average Odds Difference (AOD)
pairwise_matrix_cbs[6, 7] = 2.0 # AOD vs Statistical Parity Difference

# Fill lower triangle by reciprocity
for i in range(n):
    for j in range(i + 1, n):
        pairwise_matrix_cbs[j, i] = 1 / pairwise_matrix_cbs[i, j]


# Calculate AHP weights
eigenvalues_cbs, eigenvectors_cbs = np.linalg.eig(pairwise_matrix_cbs)
max_index_cbs = np.argmax(eigenvalues_cbs.real)
max_eigenvalue_cbs = eigenvalues_cbs[max_index_cbs].real

cbs_weights_raw = eigenvectors_cbs[:, max_index_cbs].real
cbs_weights = cbs_weights_raw / np.sum(cbs_weights_raw)

# Consistency Check
CI_cbs = (max_eigenvalue_cbs - n) / (n - 1)
RI_values = {
    1: 0.00, 2: 0.00, 3: 0.58, 4: 0.90, 5: 1.12, 6: 1.24, 7: 1.32, 8: 1.41, 9: 1.45, 10: 1.49
}
RI_cbs = RI_values[n] if n in RI_values else (1.98 if n > 10 else 1.5)
CR_cbs = CI_cbs / RI_cbs

cbs_weights_df = pd.DataFrame({
    "Metric": cbs_metrics,
    "Weight": np.round(cbs_weights, 4),
    "Percentage": np.round(cbs_weights * 100, 2)
})

print("\n" + "="*60)
print("AHP-DERIVED CBS METRIC WEIGHTS")
print("="*60)
display(cbs_weights_df)

print("\n" + "="*60)
print("CBS METRIC CONSISTENCY ANALYSIS")
print("="*60)
print(f"Maximum Eigenvalue (λmax): {max_eigenvalue_cbs:.4f}")
print(f"Consistency Index (CI): {CI_cbs:.4f}")
print(f"Consistency Ratio (CR): {CR_cbs:.4f}")

if CR_cbs < 0.10:
    print("\nConsistency Status: ACCEPTABLE\nThe weights are consistent.")
else:
    print("\nConsistency Status: NOT ACCEPTABLE\nRevise the pairwise comparison matrix for CBS metrics.")
print("="*60)


### Updated Explainability Guided Harm Assessment (EGHA) with AHP-derived CBS Weights

Now, the Combined Bias Score (CBS) will be calculated using the weights you derived above, providing a more refined and analytically supported assessment of harm.

In [ ]:
# Modify the existing EGHA calculation to use the AHP-derived CBS weights

# Map the derived weights to a dictionary for easy lookup
cbs_weights_dict = cbs_weights_df.set_index('Metric')['Weight'].to_dict()

# Redefine the CBS calculation within the EGHA loop
# ============================================================ EGHA (Modified) ============================================================

# Redefine the function or recalculate the harm results with the new CBS weights
harm_results = []

for attr in audit_attributes:

    metrics = calculate_all_metrics(
        y_test_wb,
        y_pred_xgb_wb,
        X_test_wb[attr]
    )

    # New CBS calculation using AHP-derived weights
    CBS = (
        abs(metrics['demographic_parity_difference']) * cbs_weights_dict.get("Demographic Parity Difference", 0) +
        abs(1 - metrics['disparate_impact']) * cbs_weights_dict.get("Disparate Impact", 0) +
        abs(metrics['equal_opportunity_difference']) * cbs_weights_dict.get("Equal Opportunity Difference", 0) +
        abs(metrics['selection_rate_difference']) * cbs_weights_dict.get("Selection Rate Difference", 0) +
        abs(metrics['false_positive_rate_difference']) * cbs_weights_dict.get("False Positive Rate Difference", 0) +
        abs(metrics['false_negative_rate_difference']) * cbs_weights_dict.get("False Negative Rate Difference", 0) +
        abs(metrics['average_odds_difference']) * cbs_weights_dict.get("Average Odds Difference", 0) +
        abs(metrics['statistical_parity_difference']) * cbs_weights_dict.get("Statistical Parity Difference", 0)
    )

    SIS = aggregated_sis_df.loc[
        aggregated_sis_df['Attribute'] == attr,
        'SIS'
    ].values[0]

    # ========================================================
    # GEOMETRIC HARM SCORE
    # ========================================================

    HS = np.sqrt(CBS * SIS)

    harm_results.append([
        attr,
        round(CBS,4),
        round(float(SIS),4),
        round(float(HS),4)
    ])

# Re-create harm_df with the updated CBS and HS values
harm_df = pd.DataFrame(
    harm_results,
    columns=[
        'Attribute',
        'CBS',
        'SIS',
        'HS'
    ]
)

harm_df = harm_df.sort_values(
    by='HS',
    ascending=False
)

harm_df.insert(
    0,
    'Rank',
    range(1, len(harm_df)+1)
)

print("\nATTRIBUTE HARM RANKING (Updated with AHP CBS Weights)")
display(harm_df)

# Re-classify harm based on new HS values
harm_df['Harm_Level'] = harm_df['HS'].apply(
    lambda x: "Low" if x < 0.20 else ("Moderate" if x < 0.40 else ("High" if x < 0.60 else "Critical"))
)

print("\nFINAL EGHA RESULTS (Updated)")
display(harm_df)

# Re-generate visualization
plt.figure(figsize=(10,6))

bars = plt.bar(
    harm_df['Attribute'],
    harm_df['HS']
)

for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f'{height:.3f}',
        ha='center',
        va='bottom'
    )

plt.title(
    'Explainability Guided Harm Assessment (EGHA) - Updated'
)

plt.xlabel('Sensitive Attribute')
plt.ylabel('Harm Score (HS)')

plt.xticks(rotation=30)

plt.tight_layout()

plt.show()

print("\nMOST HARMFUL ATTRIBUTE (Updated)")

print(
    f"{harm_df.iloc[0]['Attribute']}"
)

print(
    f"Harm Score = {harm_df.iloc[0]['HS']}"
)

print(
    f"Harm Level = {harm_df.iloc[0]['Harm_Level']}"
)


In [ ]:
# ============================================================
# FINAL FAIRNESS RISK INDEX (FRI)
# ============================================================

print("\n" + "="*70)
print("FINAL FAIRNESS RISK INDEX (FRI)")
print("="*70)

# ------------------------------------------------------------
# 1. DISCRIMINATION RISK (DR)
# ------------------------------------------------------------
# Using Hybrid model fairness metrics

DR = (
    abs(metrics_hybrid['demographic_parity_difference']) +
    abs(metrics_hybrid['statistical_parity_difference']) +
    abs(1 - metrics_hybrid['disparate_impact']) +
    abs(metrics_hybrid['equal_opportunity_difference'])
) / 4

# ------------------------------------------------------------
# 2. BIAS AMPLIFICATION RISK (BAR)
# ------------------------------------------------------------

dataset_dpd = 0.1161

hybrid_dpd = metrics_hybrid[
    'demographic_parity_difference'
]

BAF = abs(
    (hybrid_dpd - dataset_dpd)
    / dataset_dpd
)

BAR = BAF

# ------------------------------------------------------------
# 3. HARM INFLUENCE RISK (HIR)
# ------------------------------------------------------------

HIR = harm_df['HS'].max()

# ------------------------------------------------------------
# 4. STABILITY RISK (SR)
# ------------------------------------------------------------

behavioral_consistency = 1.00   # 100%

SR = 1 - behavioral_consistency

# ------------------------------------------------------------
# 5. FIDELITY RISK (FR)
# ------------------------------------------------------------

fidelity_score = 0.9853   # 98.53%

FR = 1 - fidelity_score

# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

DR = min(DR, 1)
BAR = min(BAR, 1)
HIR = min(HIR, 1)
SR = min(SR, 1)
FR = min(FR, 1)

# ------------------------------------------------------------
# FRI CALCULATION
# ------------------------------------------------------------

FRI = (
    0.4904 * DR +
    0.2264 * BAR +
    0.1407 * HIR +
    0.0867 * SR +
    0.0558 * FR
) * 100

# ------------------------------------------------------------
# RISK LEVEL
# ------------------------------------------------------------

if FRI < 20:
    risk_level = "LOW"

elif FRI < 40:
    risk_level = "MODERATE"

elif FRI < 60:
    risk_level = "HIGH"

else:
    risk_level = "CRITICAL"

# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

fri_df = pd.DataFrame({
    'Component': [
        'Discrimination Risk (DR)',
        'Bias Amplification Risk (BAR)',
        'Harm Influence Risk (HIR)',
        'Stability Risk (SR)',
        'Fidelity Risk (FR)'
    ],
    'Value': [
        round(DR,4),
        round(BAR,4),
        round(HIR,4),
        round(SR,4),
        round(FR,4)
    ]
})

display(fri_df)

print("\nFRI SCORE :", round(FRI,2))
print("RISK LEVEL :", risk_level)

print("="*70)

# FINAL FAIRNESS METRICS OUTPUT REPORT

In [ ]:
import subprocess
import os
from IPython.display import FileLink, display

print('Installing texlive-xetex for PDF conversion...')
!apt-get update
!apt-get install -y texlive-xetex
print('texlive-xetex installed.')

# Check if pandoc is installed, if not, install it.
print("Checking for pandoc installation...")
try:
    subprocess.run(["pandoc", "--version"], capture_output=True, check=True)
    print("Pandoc is already installed.")
except FileNotFoundError:
    print("Pandoc not found. Installing pandoc...")
    # It's generally good practice to update the package list first
    subprocess.run(["sudo", "apt-get", "update"], check=True)
    subprocess.run(["sudo", "apt-get", "install", "pandoc"], check=True)
    print("Pandoc installed successfully.")

input_md_file = "fairness_audit_report.md"
output_pdf_file = "fairness_audit_report.pdf"

# Ensure the markdown file exists before conversion
if not os.path.exists(input_md_file):
    print(f"Error: The markdown file '{input_md_file}' does not exist. Please ensure the report generation cell was run correctly.")
else:
    print(f"Converting '{input_md_file}' to '{output_pdf_file}'...")
    try:
        # Use pandoc to convert markdown to PDF
        # Add --pdf-engine=xelatex for better Unicode support and general robustness
        # Add --variable geometry:margin=1in to reduce PDF margins
        process = subprocess.run(
            ["pandoc", input_md_file, "-o", output_pdf_file, "--pdf-engine=xelatex", "--variable", "geometry:margin=1in"],
            capture_output=True, check=True, text=True
        )
        print(f"Successfully converted '{input_md_file}' to '{output_pdf_file}'.")
        print(f"Pandoc Stdout:\n{process.stdout}")
        print(f"Pandoc Stderr:\n{process.stderr}")
        display(FileLink(output_pdf_file))
    except subprocess.CalledProcessError as e:
        print(f"Error during pandoc PDF conversion: {e}")
        print(f"Stderr: {e.stderr}")
        print(f"Stdout: {e.stdout}")
        print("This might be due to issues with LaTeX installation or complex markdown content with embedded images.")
        print("If PDF conversion fails, consider simplifying markdown or exporting to DOCX/HTML first.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import io
import base64
import matplotlib.pyplot as plt
import seaborn as sns
import contextlib
import re # For regex to extract p-value
import os
# shutil is no longer needed as temp_plots directory is removed

# Helper function to safely get a variable
def get_var(name, default=None):
    return globals().get(name, default)

report_content = ""

# Add the overall report heading
report_content += "# FINAL FAIRNESS METRICS OUTPUT REPORT\n"

# SECTION 1: DATASET INFORMATION
report_content += """
## SECTION 1: Dataset Information
"""
# Use df from kernel state
if 'df' in globals() and df is not None:
    # Using original `df` for basic stats, as `df_inspect` is a copy for audit
    num_features = len(df.select_dtypes(include=np.number).columns.tolist())
    categorical_features = len(df.select_dtypes(include=['object', 'category']).columns.tolist())
    target_col_display = 'Hiring_Decision' if 'Hiring_Decision' in df.columns else 'N/A'

    report_content += f"""
- **Dataset Shape**: {df.shape[0]} rows, {df.shape[1]} columns
- **Target Column**: {target_col_display}
- **Numerical Features**: {num_features}
- **Categorical Features**: {categorical_features}
"""
    # Missing Value Summary
    missing_values = df.isnull().sum()
    missing_report = missing_values[missing_values > 0]
    if not missing_report.empty:
        report_content += "- **Missing Values**:\n"
        for idx, val in missing_report.items():
            report_content += f"  - {idx}: {val} ({val/df.shape[0]:.2%})\n"
    else:
        report_content += "- **Missing Values**: None detected.\n"
else:
    report_content += "- Dataset information not available (variable 'df' not found).\n"

# SECTION 2: REPRESENTATION BIAS
report_content += """
## SECTION 2: Representation Bias
"""
if 'imbalance_df' in globals() and imbalance_df is not None and not imbalance_df.empty:
    report_content += """
### Ranking of Most Imbalanced Groups (Higher Ratio = More Imbalanced)
"""
    report_content += imbalance_df.round(4).to_markdown(index=False) + "\n\n"
    report_content += "**Interpretation of Imbalance Severity:**\n"
    for index, row in imbalance_df.iterrows():
        report_content += f"  - '{row['Attribute']}' has an imbalance ratio of {row['Imbalance_Ratio']:.2f}, with '{row['Majority_Group']}' being the majority group and '{row['Minority_Group']}' the minority. This indicates a significant representational disparity.\n"
else:
    report_content += "- Representation bias data not available or DataFrame is empty.\n"

# SECTION 3: HISTORICAL BIAS
report_content += """
## SECTION 3: Historical Bias
"""
if 'hist_ranking_df' in globals() and hist_ranking_df is not None and not hist_ranking_df.empty:
    report_content += """
### Ranking of Attributes by Historical Bias Severity
"""
    report_content += hist_ranking_df.round(4).to_markdown(index=False) + "\n\n"
    report_content += "**Interpretation:**\n"
    for index, row in hist_ranking_df.iterrows():
        report_content += f"- '{row['Attribute']}' shows a maximum historical hiring disparity of {row['Max_Disparity']:.4f}, with '{row['Disadvantaged_Group']}' being the most disadvantaged group historically compared to '{row['Advantaged_Group']}'.\n"
else:
    report_content += "- Historical bias data not available or DataFrame is empty.\n"

# SECTION 4: INTERSECTIONAL BIAS
report_content += """
## SECTION 4: Intersectional Bias
"""
if 'intersectional_stats' in globals() and intersectional_stats is not None and not intersectional_stats.empty:
    report_content += """
### Intersectional Hiring Rates (Gender x Education Level)
"""
    report_content += intersectional_stats.round(4).to_markdown() + "\n\n"

    # Visualization via Heatmap
    fig, ax = plt.subplots(figsize=(8, 5)) # Create a figure and axes
    sns.heatmap(intersectional_stats, annot=True, fmt='.2%', cmap='YlOrRd', cbar_kws={'label': 'Hiring Rate'}, ax=ax)
    ax.set_title('Intersectional Hiring Rates: Gender vs Education Level')
    ax.set_xlabel('Education Level')
    ax.set_ylabel('Gender')
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig) # Close the figure to prevent it from being displayed separately
    intersectional_plot_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    report_content += f"![Intersectional Hiring Rates](data:image/png;base64,{intersectional_plot_base64})\n\n"

    if 'min_idx' in globals() and 'min_rate' in globals():
        report_content += f"- **Most Disadvantaged Intersectional Group**: The group {get_var('min_idx')} has the lowest historical hiring rate at {get_var('min_rate'):.2%}. This highlights areas of compounded bias.\n"
else:
    report_content += "- Intersectional bias data not available or DataFrame is empty.\n"

# SECTION 5: MODEL PERFORMANCE
report_content += """
## SECTION 5: Model Performance
"""
if 'accuracy_xgb' in globals() and 'accuracy_mlp' in globals() and 'accuracy_hybrid' in globals() and 'accuracy_rf' in globals() and 'accuracy_lgbm' in globals():
    performance_data = {
        'Model': ['XGBoost', 'MLP Neural Network', 'Hybrid Model', 'Random Forest', 'LightGBM'],
        'Accuracy': [accuracy_xgb, accuracy_mlp, accuracy_hybrid, accuracy_rf, accuracy_lgbm]
    }
    performance_df = pd.DataFrame(performance_data)
    report_content += """
### Overall Model Accuracy Comparison
"""
    report_content += performance_df.round(4).to_markdown(index=False) + "\n\n"
    best_model = performance_df.loc[performance_df['Accuracy'].idxmax()]
    worst_model = performance_df.loc[performance_df['Accuracy'].idxmin()]
    report_content += f"- **Best Performing Model**: {best_model['Model']} with an Accuracy of {best_model['Accuracy']:.4f}.\n"
    report_content += f"- **Worst Performing Model**: {worst_model['Model']} with an Accuracy of {worst_model['Accuracy']:.4f}.\n"
else:
    report_content += "- Model performance metrics not fully available.\n"

# SECTION 6: WHITE-BOX FAIRNESS
report_content += """
## SECTION 6: White-Box Fairness
"""
if 'fairness_results_df' in globals() and fairness_results_df is not None and not fairness_results_df.empty:
    report_content += """
### Fairness Metrics Comparison (DPD, DI, EOD) for White-Box Models
"""
    # Ensure 'Abs_SPD' is calculated if not already present
    if 'Statistical Parity Difference (SPD)' in fairness_results_df.columns and 'Abs_SPD' not in fairness_results_df.columns:
        fairness_results_df['Abs_SPD'] = fairness_results_df['Statistical Parity Difference (SPD)'].abs()

    report_content += fairness_results_df.round(4).to_markdown(index=False) + "\n\n"
    # Determine most biased/fair models based on SPD
    # Assuming lower absolute SPD is more fair
    if 'Abs_SPD' in fairness_results_df.columns:
        most_fair_model_spd = fairness_results_df.loc[fairness_results_df['Abs_SPD'].idxmin()]
        most_biased_model_spd = fairness_results_df.loc[fairness_results_df['Abs_SPD'].idxmax()]
        report_content += f"- **Most Fair Model (by SPD)**: {most_fair_model_spd['Model']} (SPD: {most_fair_model_spd['Statistical Parity Difference (SPD)']:.4f})\n"
        report_content += f"- **Most Biased Model (by SPD)**: {most_biased_model_spd['Model']} (SPD: {most_biased_model_spd['Statistical Parity Difference (SPD)']:.4f})\n"
    else:
        report_content += "- Cannot rank models by fairness; 'Statistical Parity Difference (SPD)' column not found.\n"
else:
    report_content += "- White-box fairness results not available or DataFrame is empty.\n"

# SECTION 7: BIAS AMPLIFICATION IN WHITE BOX MODEL
report_content += """
## SECTION 7: Bias Amplification in white box model
"""
if 'whitebox_comparison_df' in globals() and whitebox_comparison_df is not None and not whitebox_comparison_df.empty:
    report_content += """
### WHITE-BOX FAIRNESS COMPARISON TABLE
"""
    report_content += whitebox_comparison_df.round(4).to_markdown(index=False) + "\n\n"
    report_content += f"BEST FAIR MODEL : {best_model_name}\n\n"
    report_content += f"LOWEST DPD      : {best_model_dpd:.4f}\n\n"

if 'amp_df' in globals() and amp_df is not None and not amp_df.empty:
    report_content += """
### Bias Amplification Ranking (Positive = Model increased bias)
"""
    report_content += amp_df.round(4).to_markdown(index=False) + "\n\n"
    positive_amplification = amp_df[amp_df['Bias_Amplification'] > 0]
    if not positive_amplification.empty:
        report_content += "**Models Worsening Discrimination:**\n"
        for idx, row in positive_amplification.iterrows():
            report_content += f"- The '{row['Attribute']}' attribute shows bias amplification (increase) of {row['Bias_Amplification']:.4f}, indicating the White-Box XGBoost model worsened this disparity.\n"
    else:
        report_content += "- No attributes showed positive bias amplification by the White-Box XGBoost model. Bias was generally mitigated or remained similar.\n"
else:
    report_content += "- Bias amplification data not available or DataFrame is empty.\n"

# SECTION 8: BLACK-BOX FAIRNESS
report_content += """
## SECTION 8: Black-Box Fairness
"""
if 'final_report' in globals() and final_report is not None:
    if 'Surrogate Model Performance' in final_report:
        report_content += """
### Surrogate Model Performance
"""
        for metric, value in final_report['Surrogate Model Performance'].items():
            report_content += f"- {metric}: {value}\n\n"
    if 'Fairness Metrics' in final_report:
        report_content += """
### Fairness Metrics (Black-Box Surrogate)
"""
        for attr, metrics in final_report['Fairness Metrics'].items():
            report_content += f"**{attr}:**\n"
            for metric, value in metrics.items():
                report_content += f"- {metric}: {value}\n"
    if 'Group Disparity Analysis' in final_report:
        report_content += """
### Group Disparity Analysis
"""
        for key, value in final_report['Group Disparity Analysis'].items():
            # Handle dict values specifically for selection rates
            if isinstance(value, dict):
                report_content += f"- **{key}**: " + pd.DataFrame(value).T.round(2).to_markdown(index=False) + "\n"
            else:
                report_content += f"- **{key}**: {value}\n"
else:
    report_content += "- Black-box fairness analysis (final_report variable) not available.\n"

if 'fidelity_score' in globals() and 'c_score' in globals():
    report_content += f"- **Surrogate Model Fidelity Score**: {fidelity_score:.2%}\n"
    report_content += f"- **Behavioral Consistency Score**: {c_score:.2%}\n"
    report_content += "- **Instability Interpretation**: A high consistency score (e.g., near 100%) indicates that small perturbations to non-sensitive features do not frequently flip the model's decision, suggesting robust and stable behavior. A lower score would indicate instability.\n"
else:
    report_content += "- Fidelity or Consistency scores not fully available.\n"

if 'sensitivity_df' in globals() and sensitivity_df is not None and not sensitivity_df.empty:
    report_content += """
### Sensitivity Analysis Ranking (Top 10)
"""
    report_content += sensitivity_df.head(10).round(4).to_markdown(index=False) + "\n\n"
    report_content += "- **Hidden Discrimination Interpretation**: Features with high sensitivity might indicate that the model is overly reliant on them, and if these features are correlated with sensitive attributes, they could act as proxies for discrimination. Careful examination of highly sensitive features is warranted.\n"
else:
    report_content += "- Sensitivity analysis data not available or DataFrame is empty.\n"

# SECTION 9: PRECISE BIAS PROGRESSION AUDIT
report_content += """
## SECTION 9: Precise Bias Progression Audit
"""
if 'precision_df' in globals() and precision_df is not None and not precision_df.empty:
    report_content += """
### Detailed Bias Shifts Across Recruitment Stages (Demographic Parity Difference)
"""
    report_content += precision_df.to_markdown(index=False) + "\n\n"

    # --- Plot: Evolution of Absolute Demographic Parity Difference Across System Stages ---
    if 'dpd_evolution_df' in globals() and dpd_evolution_df is not None and not dpd_evolution_df.empty:
        fig, ax = plt.subplots(figsize=(8, 4)) # Create a figure and axes
        sns.lineplot(data=dpd_evolution_df, x='Stage', y='DPD', hue='Attribute', marker='o', palette='tab10', ax=ax)
        ax.set_title('Evolution of Absolute Demographic Parity Difference Across System Stages')
        ax.set_xlabel('System Stage')
        ax.set_ylabel('Absolute Demographic Parity Difference (DPD)')
        ax.grid(True, linestyle='--', alpha=0.7)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight')
        plt.close(fig) # Close the figure to prevent it from being displayed separately
        dpd_evolution_plot_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
        report_content += f"![DPD Evolution](data:image/png;base64,{dpd_evolution_plot_base64})\n\n"
        report_content += "This plot illustrates the evolution of Demographic Parity Difference for each sensitive attribute across different stages of the system, from the raw dataset to the black-box surrogate model. It helps visualize where bias is introduced, amplified, or mitigated.\n\n"
    else:
        report_content += "- DPD Evolution plot data not available or DataFrame is empty.\n"

    report_content += "**Qualitative Insights:**\n"
    for index, row in precision_df.iterrows():
        # Re-calculating deltas for qualitative insights as they are not stored directly in row
        try:
            d_dpd_val = float(row['Initial (Dataset)'])
            # Use 'Blackbox Surrogate Model DPD' instead of 'Final DPD'
            s_dpd_val = float(row['Blackbox Surrogate Model DPD'])
            delta_dataset_xgb_val = float(row['XGBoost Shift'])
            # delta_xgb_mlp_val = float(row['MLP Shift']) # Removed
            delta_xgb_surrogate_val = float(row['XGBoost to System Shift (BB)'])

            # deltas = [abs(delta_dataset_xgb_val), abs(delta_xgb_mlp_val), abs(delta_mlp_surrogate_val)] # Removed
            # stages = ['Dataset to XGBoost', 'XGBoost to MLP', 'MLP to System (BB)'] # Removed

            # Updated deltas and stages for the qualitative insights
            deltas_for_qualitative = [abs(delta_dataset_xgb_val), abs(delta_xgb_surrogate_val)]
            stages_for_qualitative = ['Dataset to XGBoost', 'XGBoost to System (BB)']

            max_delta_idx = np.argmax(deltas_for_qualitative)
            largest_shift_stage = stages_for_qualitative[max_delta_idx]

            net_impact_str = "Mitigated" if (s_dpd_val - d_dpd_val) < 0 else "Amplified"
            report_content += f"* {row['Attribute']}: The system {net_impact_str} bias by {abs(d_dpd_val - s_dpd_val):.4f}. The largest shift occurred during the {largest_shift_stage} stage.\n"
        except Exception as e:
            report_content += f"* Could not generate qualitative insight for {row['Attribute']}: {e}\n"
else:
    report_content += "- Precise bias progression audit data not available or DataFrame is empty.\n"

# SECTION 10: COUNTERFACTUAL FAIRNESS
report_content += """
## SECTION 10: Counterfactual Fairness
"""
if 'd_rate' in globals():
    report_content += f"- **Counterfactual Discrimination Rate**: {d_rate:.2%}\n"
    if d_rate > 0.05:
        report_content += "- **Discrimination Severity Summary**: A counterfactual discrimination rate above 5% typically indicates a significant risk of individual discrimination, where the model's decision changes merely by altering a sensitive attribute while keeping other factors constant. Remediation is advised.\n"
    else:
        report_content += "- **Discrimination Severity Summary**: The low counterfactual discrimination rate suggests that the model rarely changes its decision solely based on alterations to sensitive attributes, indicating good individual fairness. The model passes this test.\n"
else:
    report_content += "- Counterfactual discrimination rate not available.\n"



# SECTION 11: SHAP EXPLAINABILITY
report_content += """
## SECTION 11: SHAP Explainability
"""
if 'all_models_shap_comparison' in globals() and all_models_shap_comparison is not None and not all_models_shap_comparison.empty:
    report_content += """
### Mean Absolute SHAP values across XGBoost, MLP, and Surrogate Models (Top 15)"""
    report_content += all_models_shap_comparison.head(15).round(4).to_markdown(index=False) + "\n\n"
    report_content += "- **Sensitive Feature Influence Summary**: Observing the SHAP values for sensitive attributes (e.g., 'Gender', 'Education_Level') in this table helps determine their direct influence on model predictions. If they appear high, it indicates direct reliance. Cross-model comparison can reveal if this influence is consistent or model-specific.\n"
else:
    report_content += "- Combined SHAP comparison data not available or DataFrame is empty.\n"

if 'shap_importance' in globals() and shap_importance is not None and not shap_importance.empty:
    report_content += """
### Top 10 Most Influential Features (Surrogate Model SHAP)
"""
    report_content += shap_importance.head(10).round(4).to_markdown(index=False) + "\n\n"
    # Add the plot embedding here
    try:
        fig = plt.figure(figsize=(6, 6))
        sns.barplot(x='SHAP_Importance', y='Feature', data=shap_importance.head(10), palette='viridis', hue='Feature', legend=False)
        plt.title('Top 10 Most Influential Features (Surrogate Model SHAP)')
        plt.xlabel('Mean Absolute SHAP Value')
        plt.ylabel('Feature')
        plt.tight_layout()
        buf = io.BytesIO()
        plt.savefig(buf, format='png', bbox_inches='tight')
        plt.close(fig) # Close the figure to prevent it from being displayed separately
        shap_plot_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
        report_content += f"![Top 10 Most Influential Features](data:image/png;base64,{shap_plot_base64})\n\n"
    except Exception as e:
        report_content += f"- Could not generate SHAP importance plot: {e}\n"
else:
    report_content += "- Surrogate model SHAP importance data not available or DataFrame is empty.\n"

# SECTION 12: Explainability-Guided Harm Assessment (EGHA)
report_content += """
## SECTION 12: Explainability-Guided Harm Assessment (EGHA)
"""
if 'harm_df' in globals() and harm_df is not None and not harm_df.empty:
    report_content += """
### Attribute Harm Ranking (Explainability-Guided Harm Assessment)
"""
    report_content += harm_df.round(4).to_markdown(index=False) + "\n\n"
    report_content += "**Most Harmful Attribute:**\n"
    most_harmful_attr = harm_df.iloc[0]
    report_content += f"- **Attribute**: {most_harmful_attr['Attribute']}\n"
    report_content += f"- **Harm Score (HS)**: {most_harmful_attr['HS']:.4f}\n"
    report_content += f"- **Harm Level**: {most_harmful_attr['Harm_Level']}\n"
else:
    report_content += "- Explainability-Guided Harm Assessment (EGHA) data not available or DataFrame is empty.\n"


# SECTION 13: STATISTICAL VALIDATION
report_content += """
## SECTION 13: Statistical Validation
"""

if 'audit_results_df_cleaned' in globals() and audit_results_df_cleaned is not None and not audit_results_df_cleaned.empty:
    report_content += """
### Statistical Validation of Inherent Dataset Bias

This visualization displays the mean inherent Statistical Parity Difference (SPD) with 95% Bootstrap Confidence Intervals for each sensitive attribute, along with their p-values and statistical significance status. This helps in understanding which biases in the raw dataset are statistically significant and their estimated magnitude.
"""

    report_content += """
#### Detailed Statistical Validation Results for All Attributes
"""
    report_content += audit_results_df_cleaned[['attribute', 'chi2', 'p_value', 'significance_status', 'mean_spd', 'boot_ci_lower', 'boot_ci_upper']].round(4).to_markdown(index=False) + "\n\n"

    fig, ax = plt.subplots(figsize=(7.5, 6)) # Create a figure and axes
    sns.barplot(
        x='attribute',
        y='mean_spd',
        data=audit_results_df_cleaned,
        palette='viridis',
        hue='attribute', # Added to resolve FutureWarning
        legend=False,     # Added to resolve FutureWarning
        ax=ax
    )

    for i, row in audit_results_df_cleaned.iterrows():
        y_upper_bound = row['mean_spd'] + row['yerr_upper'] if pd.notna(row['yerr_upper']) else row['mean_spd']
        y_offset = y_upper_bound + 0.01

        ax.errorbar(
            x=i,
            y=row['mean_spd'],
            yerr=[[row['yerr_lower']], [row['yerr_upper']]],
            fmt='none',
            c='black',
            capsize=5,
            elinewidth=1
        )
        ax.text(
            i,
            y_offset,
            f"P: {row['p_value']:.2e}\n({row['significance_status']})",
            color='black',
            ha='center',
            va='bottom',
            fontsize=9,
            weight='bold' if row['significance_status'] == 'Significant' else 'normal'
        )

    ax.set_title('Mean Inherent Statistical Parity Difference (SPD) with 95% CI')
    ax.set_xlabel('Sensitive Attribute')
    ax.set_ylabel('Mean Inherent SPD (Max-Min Selection Rate)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig) # Close the figure to prevent it from being displayed separately
    statistical_validation_plot_base64 = base64.b64encode(buf.getvalue()).decode('utf-8')
    report_content += f"![Statistical Validation](data:image/png;base64,{statistical_validation_plot_base64})\n\n"
    report_content += "- **Fairness Reliability Interpretation**: This visualization confirms the statistical reliability of the observed disparities and metrics, providing confidence in the audit findings. Attributes marked 'Significant' have disparities unlikely due to random chance.\n"
else:
    report_content += "- Statistical validation results (audit_results_df_cleaned) not available or DataFrame is empty.\n"


# SECTION 14: Final Fairness Risk Scoring
report_content += """
## SECTION 14: Final Fairness Risk Scoring
"""
# Use FRI and risk_level from kernel state
if 'FRI' in globals() and 'risk_level' in globals():
    report_content += f"- **Overall Fairness Risk Index (FRI) Score**: {FRI:.2f} / 100\n"
    report_content += f"- **Overall Risk Level**: {risk_level}\n"
else:
    report_content += "- Final Fairness Risk Index (FRI) Score not available.\n"

if 'combined_bias_df' in globals() and 'combined_bias_threshold' in globals() and combined_bias_df is not None and not combined_bias_df.empty:
    high_priority_attributes = combined_bias_df[combined_bias_df['Combined_DPD_Bias_Score'] >= combined_bias_threshold]
    if not high_priority_attributes.empty:
        report_content += """
- **Fairness Debt Alerts (High Priority Attributes)**: The following attributes exceeded the Combined Bias Threshold of {combined_bias_threshold:.2f} and require immediate intervention:""" + "\n"
        report_content += high_priority_attributes.to_markdown(index=False) + "\n"
    else:
        report_content += f"- **Fairness Debt Alerts**: No attributes exceeded the Combined Bias Threshold of {combined_bias_threshold:.2f}.\n"
else:
    report_content += "- Combined Bias Score data for risk alerts not fully available.\n"


# SECTION 15: FINAL RESEARCH CONCLUSION
report_content += """
## SECTION 15: Final Research Conclusion
"""

# Dynamically retrieve values for the conclusion
# Use get_var for safety and to reuse existing kernel variables

# DPD values for conclusion
dpd_gender_dataset = get_var('all_attributes_comparison_data', {}).get('Gender', {}).get('Inherent Dataset Bias', {}).get('Demographic Parity Difference', 'N/A')
dpd_education_dataset = get_var('all_attributes_comparison_data', {}).get('Education_Level', {}).get('Inherent Dataset Bias', {}).get('Demographic Parity Difference', 'N/A')
# Use first two highest DPD attributes from combined_bias_df for consistency with context
highest_dpd_attrs_conclusion = get_var('combined_bias_df', pd.DataFrame()).iloc[0]['Attribute'] if not get_var('combined_bias_df', pd.DataFrame()).empty else 'N/A'
second_highest_dpd_attrs_conclusion = get_var('combined_bias_df', pd.DataFrame()).iloc[1]['Attribute'] if len(get_var('combined_bias_df', pd.DataFrame())) > 1 else 'N/A'

# Job_Role_Applied DPD amplification values
job_role_initial = get_var('all_attributes_comparison_data', {}).get('Job_Role_Applied', {}).get('Inherent Dataset Bias', {}).get('Demographic Parity Difference', 'N/A')
job_role_final = get_var('all_attributes_comparison_data', {}).get('Job_Role_Applied', {}).get('Black-Box Surrogate', {}).get('Demographic Parity Difference', 'N/A')

# Most fair/biased model names from kernel state
most_fair_model_name = get_var('most_fair_model_spd', {}).get('Model', 'N/A')
most_biased_model_name = get_var('most_biased_model_spd', {}).get('Model', 'N/A')

# Intersectional group
min_intersectional_group = get_var('min_idx', 'N/A')
min_intersectional_rate = get_var('min_rate', 'N/A')

# Counterfactual discrimination rate and severity score for conclusion
d_rate_conclusion = get_var('d_rate', 'N/A')
# Use FRI for conclusion
FRI_conclusion = get_var('FRI', 'N/A')

# Ensure all numeric values are formatted, and N/A handled gracefully
# Also ensure correct attribute names are used from `combined_bias_df` for DPD values.
# Re-fetch DPD values for specific attributes used in conclusion for robustness if they differ from general `dpd_gender_dataset`/`dpd_education_dataset` based on how they are calculated in context

# Based on the kernel state, highest_dpd_attrs_conclusion is 'Previous_Companies' (0.1542) and second_highest_dpd_attrs_conclusion is 'Education_Level' (0.1102).
# Let's use these dynamic values directly.

dpd_prev_companies_dataset = get_var('all_attributes_comparison_data', {}).get('Previous_Companies', {}).get('Inherent Dataset Bias', {}).get('Demographic Parity Difference', 'N/A')
dpd_edu_level_dataset = get_var('all_attributes_comparison_data', {}).get('Education_Level', {}).get('Inherent Dataset Bias', {}).get('Demographic Parity Difference', 'N/A')


report_content += f"""
Based on the comprehensive fairness audit, the following key conclusions can be drawn:

1.  **Dataset Discrimination**: The dataset contains inherent discrimination, particularly in '{highest_dpd_attrs_conclusion}' (DPD ~{dpd_prev_companies_dataset:.3f} if dpd_prev_companies_dataset != 'N/A' else 'N/A') and '{second_highest_dpd_attrs_conclusion}' (DPD ~{dpd_edu_level_dataset:.3f} if dpd_edu_level_dataset != 'N/A' else 'N/A'), confirmed by historical bias analysis and statistical validation. This indicates pre-existing human prejudice within the raw data.

2.  **Model Bias Amplification/Mitigation**: For most sensitive attributes (e.g., 'Gender', 'Education_Level'), a trend of **DPD mitigation** is observed as we move from the raw dataset through the white-box models to the black-box surrogate model. This suggests that the combined system generally reduces some inherent biases.
    However, for 'Job_Role_Applied', the DPD is **amplified** in the Black-Box Surrogate model (increasing from ~{job_role_initial:.3f} to ~{job_role_final:.3f} if isinstance(job_role_initial, (int, float)) and isinstance(job_role_final, (int, float)) else 'N/A'). This indicates that the black-box system's decision-making process, including expert rules, may exacerbate disparities related to job roles.

3.  **Most Fair Model**: Among the white-box models, **{most_fair_model_name}** generally shows relatively lower Statistical Parity Difference (SPD), suggesting it is comparably fair in terms of selection rates. The black-box surrogate also shows significant mitigation for some attributes like Gender, making the overall system behavior more fair in those aspects.

4.  **Most Biased Model**: **{most_biased_model_name}** appears to have the highest SPD, indicating it is comparatively more biased regarding selection rates for its most biased attribute. However, specific amplification of 'Job_Role_Applied' in the black-box system is also a critical concern.

5.  **Most Vulnerable Demographic Groups**: Based on combined bias scores and intersectional analysis, '{highest_dpd_attrs_conclusion}' and '{second_highest_dpd_attrs_conclusion}' consistently show the highest DPD. Intersectional groups, such as {min_intersectional_group} candidates, exhibit significantly lower hiring rates (at {min_intersectional_rate:.2%} if min_intersectional_rate != 'N/A' else 'N/A'), identifying them as particularly vulnerable to compounded biases.

6.  **Hidden Bias in Black-Box System**: While the black-box system mitigates bias for several attributes, the amplification of DPD for 'Job_Role_Applied' points to hidden biases within its composite logic. SHAP analysis on the surrogate model also highlights influential features that may implicitly act as proxies.

7.  **Counterfactual Discrimination**: The low counterfactual discrimination rate ({d_rate_conclusion:.2%} if d_rate_conclusion != 'N/A' else 'N/A') suggests the system generally passes individual fairness tests, where altering a protected attribute does not disproportionately flip decisions for an individual.

8.  **Statistical Reliability**: Fairness metrics are statistically reliable, as confirmed by Chi-Square tests (e.g., for Gender) and bootstrap confidence intervals, affirming that observed disparities are statistically significant and not due to random chance.

9.  **Ethical Implications**: The audit reveals the system, while mitigating some biases, has critical areas needing attention, especially the amplification of bias for 'Job_Role_Applied' and persistent disparities in '{highest_dpd_attrs_conclusion}' and '{second_highest_dpd_attrs_conclusion}'. Ethical deployment requires targeted interventions to address these issues and ensure equitable hiring outcomes.

10. **Research Contributions**: This research provides a robust framework for multi-stage fairness auditing, combining white-box and black-box analysis with explainability (SHAP) and statistical validation. It demonstrates the importance of a holistic approach to identify and characterize bias evolution throughout the AI system lifecycle, offering actionable insights for responsible AI development and deployment.
"""

# Print the entire report content
display(Markdown(report_content))

report_filename = "fairness_audit_report.md"
with open(report_filename, "w") as f:
    f.write(report_content)

In [ ]:
import subprocess
import os
from IPython.display import FileLink, display

print('Updating apt-get and installing texlive-xetex for PDF conversion...')
!apt-get update
!apt-get install -y texlive-xetex
print('texlive-xetex installed.')

# Check if pandoc is installed, if not, install it.
print("Checking for pandoc installation...")
try:
    # Use 'which' command to check if pandoc is in PATH
    subprocess.run(["which", "pandoc"], capture_output=True, check=True)
    print("Pandoc is already installed.")
except subprocess.CalledProcessError:
    print("Pandoc not found. Installing pandoc...")
    !apt-get install -y pandoc
    print("Pandoc installed successfully.")

input_md_file = "fairness_audit_report.md"
output_pdf_file = "fairness_audit_report.pdf"

# Ensure the markdown file exists before conversion
if not os.path.exists(input_md_file):
    print(f"Error: The markdown file '{input_md_file}' does not exist. Please ensure the report generation cell was run correctly.")
else:
    print(f"Converting '{input_md_file}' to '{output_pdf_file}'...")
    try:
        # Use pandoc to convert markdown to PDF
        # Add --pdf-engine=xelatex for better Unicode support and general robustness
        # Add --variable geometry:margin=1in to reduce PDF margins
        process = subprocess.run(
            ["pandoc", input_md_file, "-o", output_pdf_file, "--pdf-engine=xelatex", "--variable", "geometry:margin=1in"],
            capture_output=True, check=True, text=True
        )
        print(f"Successfully converted '{input_md_file}' to '{output_pdf_file}'.")
        print(f"Pandoc Stdout:\n{process.stdout}")
        print(f"Pandoc Stderr:\n{process.stderr}")
        display(FileLink(output_pdf_file))
    except subprocess.CalledProcessError as e:
        print(f"Error during pandoc PDF conversion: {e}")
        print(f"Stderr: {e.stderr}")
        print(f"Stdout: {e.stdout}")
        print("This might be due to issues with LaTeX installation or complex markdown content with embedded images.")
        print("If PDF conversion fails, consider simplifying markdown or exporting to DOCX/HTML first.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

### Final Confirmation of Report Updates

The `fairness_audit_report.pdf` has been successfully generated and reviewed. The "Precise Bias Progression Audit" section now reflects the following changes:

*   The 'MLP Shift' column has been removed from the table.
*   The 'System Shift (BB)' column has been renamed to 'XGBoost to System Shift (BB)'.
*   The values in the 'XGBoost to System Shift (BB)' column have been recalculated to represent the DPD shift directly from the White-Box XGBoost model to the Black-Box Surrogate model.
*   The qualitative insights in this section have been updated to align with these changes, accurately reflecting the net impact (mitigation or amplification) and identifying the stage with the largest shift in DPD for each sensitive attribute based on the new calculations.

All modifications requested in the plan have been successfully implemented in the final report.